In [0]:
# XGBoost (gradient boosting) and statsmodels (GLMs with standard errors / p-values) are not in the base environment
%pip install -q xgboost statsmodels
dbutils.library.restartPython()

In [0]:
import pandas as pd

li_data = [
    'addresses',
    'claims',
    'customers',
    'policies_buildings',
    'policies_contents',
    'policies_motor',
    'sample_submission',
    'test_buildings',
    'test_contents',
    'test_motor',
    'weather'
]

kw_data = {}  

for fname in li_data:
    kw_data[fname] = pd.read_parquet(f'../data/{fname}.parquet')

In [0]:
import os
os.listdir('../data')


In [0]:
# Preview the first rows of selected datasets
for name in ['claims', 'policies_buildings', 'sample_submission', 'test_buildings']:
    df = kw_data[name]
    print(f'{name}: {df.shape[0]:,} rows x {df.shape[1]} columns')
    display(df.head(5))

### Initial data observations

| Dataset | Rows | Columns |
| --- | --- | --- |
| claims | 163,150 | 12 |
| policies_buildings | 707,726 | 27 |
| sample_submission | 626,639 | 2 |
| test_buildings | 184,155 | 26 |

* **Target = claim prediction**: the `prediction` column in the submission is the expected **ultimate claim cost** per row (all claims with a `loss_date` in the row's period, net of excess and recoveries, excluding repudiated claims). This is the risk premium, not the premium charged, and it is scored with Tweedie D² (power 1.5). It has to be built from the `claims` file, because there is no target column in the policy files.
* **`renewal_decision` is not the target**: it is in the training files only (`test_buildings` has every other `policies_buildings` column). It is known only after the period, so exclude it from features to avoid leakage.
* **Submission covers all three lines**: `sample_submission` rows = test buildings + contents + motor (184,155 + 187,498 + 254,986 = 626,639). The sample row_ids start with `M` (motor) and `prediction` is numeric.
* **`claims` spans multiple lines**: it has a `line` column (e.g. `motor`) and joins to policies on `policy_id`, so filter to buildings claims before joining.
* **`construction_type` needs cleaning**: inconsistent labels such as `Brick`, `stone`, `T`, `W`, `NON_STANDARD`.
* **Same policies recur across years**: e.g. `B89645301` appears in 2021 training data and 2025 test data, so watch for leakage when building validation splits.

### Checking the `policy_id` prefix

The first letter of `policy_id` appears to encode the product line (e.g. `M` = motor, `B` = buildings). Before relying on this, the cell below:

* Lists which raw datasets contain a `policy_id` column (`addresses`, `customers`, `weather` and `sample_submission` do not).
* Counts the first letter of `policy_id` in each dataset. Each `policies_*` and `test_*` file should contain only one letter.
* Cross-tabulates the prefix against the `line` column in `claims` as an independent check: if the prefix encodes the line, each letter should map to exactly one `line` value.

In [0]:
# Count first letter of policy_id in every dataset that has the column
prefix_summary = []
for name, df in kw_data.items():
    if 'policy_id' not in df.columns:
        print(f'{name}: no policy_id column')
        continue
    counts = df['policy_id'].str[0].value_counts(dropna=False)
    for prefix, n in counts.items():
        prefix_summary.append({'dataset': name, 'prefix': prefix, 'rows': n})

display(pd.DataFrame(prefix_summary))

# Independent check: prefix vs the claims 'line' column
display(pd.crosstab(kw_data['claims']['policy_id'].str[0], kw_data['claims']['line']).reset_index())

### Adding a `policy_type` column

The check above confirms the prefix maps one-to-one to the product line:

| Prefix | policy_type |
| --- | --- |
| `B` | buildings |
| `C` | contents |
| `M` | motor |

The cell below builds a new dictionary, `kw_data_typed`, with a copy of every raw dataset. The originals in `kw_data` are not changed. For each dataset that has a `policy_id` column, it adds `policy_type` by mapping the first letter of `policy_id` with the table above. Datasets without `policy_id` are copied unchanged. As a safeguard, the cell stops with an error if any `policy_id` has a prefix that is not in the mapping.

In [0]:
# Map policy_id prefix to product line
policy_type_map = {'B': 'buildings', 'C': 'contents', 'M': 'motor'}

kw_data_typed = {}
for name, df in kw_data.items():
    df_new = df.copy()
    if 'policy_id' in df_new.columns:
        df_new['policy_type'] = df_new['policy_id'].str[0].map(policy_type_map)
        n_unmapped = df_new['policy_type'].isna().sum()
        assert n_unmapped == 0, f'{name}: {n_unmapped} rows with unknown policy_id prefix'
    kw_data_typed[name] = df_new

# Summary of policy_type per dataset
summary = [
    {'dataset': name, 'policy_type': ptype, 'rows': n}
    for name, df in kw_data_typed.items() if 'policy_type' in df.columns
    for ptype, n in df['policy_type'].value_counts().items()
]
display(pd.DataFrame(summary))

### Adding an `earned_premium` column

`written_premium` is the full premium charged for the policy period. `exposure` is the fraction of a year the policy was actually in force (e.g. `1.0` = a full year, `0.3452` = about 4 months). Multiplying the two gives the premium **earned** over the time the policy was in force:

`earned_premium = exposure * written_premium`

The cell below loops over every dataset in `kw_data_typed` that has an `exposure` column and adds `earned_premium`. If a dataset has `exposure` but no `written_premium`, it is skipped and listed so it can be checked. The summary table shows total exposure, written premium and earned premium per dataset as a sense check: earned premium should never be more than written premium when exposure is at most 1.

In [0]:
# Add earned_premium = exposure * written_premium wherever exposure exists
ep_summary = []
for name, df in kw_data_typed.items():
    if 'exposure' not in df.columns:
        continue
    if 'written_premium' not in df.columns:
        print(f'{name}: has exposure but no written_premium - skipped')
        continue
    df['earned_premium'] = df['exposure'] * df['written_premium']
    ep_summary.append({
        'dataset': name,
        'rows': len(df),
        'exposure_min': df['exposure'].min(),
        'exposure_max': df['exposure'].max(),
        'total_written_premium': round(df['written_premium'].sum(), 2),
        'total_earned_premium': round(df['earned_premium'].sum(), 2),
        'null_earned_premium': int(df['earned_premium'].isna().sum()),
    })

display(pd.DataFrame(ep_summary))

### Data quality note: `exposure` greater than 1 in training data

The sense check above fails for the training files: total `earned_premium` is **higher** than total `written_premium`, because `exposure` goes up to 365 there. The test files are fine (max exposure = 1).

Comparing `exposure` with the policy period length (`days = period_end - period_start + 1`) shows that every training row uses one of three conventions:

| Convention | How to recognise it | Example (full-year policy) |
| --- | --- | --- |
| **Years, 365-day basis** (the standard, and all test rows) | `exposure = days / 365` | `1.0` |
| **Days instead of years** | `exposure = days` | `365.0` |
| **Years, 360-day basis** | `exposure = days / 360` | `1.0139` (= 365/360) |

* About 0.4% of training rows have `exposure > 1`: 3,044 buildings, 3,088 contents and 3,991 motor rows. These are a mix of the day-count rows (e.g. `365.0`, `327.0`) and 360-day-basis full-year rows (`1.0139`).
* The problem is **not only** rows above 1: a further 1,138 buildings, 1,222 contents and 1,745 motor rows are on the 360-day basis with short periods, so their exposure is at most 1 but still about 1.4% too high. A simple "cap at 1" would miss these.
* In total, the training rows needing conversion are:

| Dataset | Stored in days | 360-day basis | Already 365-day basis |
| --- | --- | --- | --- |
| policies_buildings | 656 | 3,526 | 703,544 |
| policies_contents | 729 | 3,582 | 717,306 |
| policies_motor | 934 | 4,802 | 974,232 |

* One contents row is a 1-day policy stored in days (`exposure = 1`), which looks like a valid full year.
* No `exposure` values are null, and period lengths range from 1 to 365 days.

The raw `earned_premium` in the cell above overstates premium for these rows. The next cells standardise exposure to the 365-day year basis and recalculate `earned_premium`.

### Standardising exposure and recalculating `earned_premium`

The cell below adds `exposure_standardised` (fraction of a 365-day year) to every dataset that has `exposure`, `period_start` and `period_end`:

1. Calculate `days = period_end - period_start + 1`.
2. Classify each row's convention, checked in this order:
   * `year_365`: `exposure` matches `days / 365` (within 4-decimal rounding). Kept as is.
   * `days`: `exposure` equals `days` (and is at least 1). Converted to `days / 365`.
   * `year_360`: `exposure` matches `days / 360`. Converted to `days / 365`.

   Converted rows use `days / 365` directly rather than rescaling the stored value, because the stored 360-basis values are rounded to 4 decimals (e.g. `1.0139 * 360 / 365` gives 1.00001, not exactly 1).
3. Stop with an error if any row matches none of these, so unexpected values are never silently converted.
4. Overwrite `earned_premium` with `exposure_standardised * written_premium`. The raw `exposure` column is kept unchanged.

The summary table shows how many rows were converted from each convention, and checks that `exposure_standardised` is now between 0 and 1 and that total `earned_premium` is no longer more than total `written_premium`.

In [0]:
import numpy as np

TOL = 0.00006  # exposure is stored to 4 decimal places

std_summary = []
for name, df in kw_data_typed.items():
    if not {'exposure', 'period_start', 'period_end'}.issubset(df.columns):
        continue

    days = (pd.to_datetime(df['period_end']) - pd.to_datetime(df['period_start'])).dt.days + 1
    exp = df['exposure']

    is_365 = np.isclose(exp, days / 365, atol=TOL, rtol=0)
    is_days = ~is_365 & (exp >= 1) & np.isclose(exp, days, atol=0.5, rtol=0)
    is_360 = ~is_365 & ~is_days & np.isclose(exp, days / 360, atol=TOL, rtol=0)
    unmatched = ~(is_365 | is_days | is_360)
    assert unmatched.sum() == 0, f'{name}: {unmatched.sum()} rows with unrecognised exposure'

    df['exposure_standardised'] = np.select(
        [is_365, is_days, is_360],
        [exp, days / 365, days / 365],  # converted rows use the exact period length
    )
    df['earned_premium'] = df['exposure_standardised'] * df['written_premium']

    std_summary.append({
        'dataset': name,
        'rows': len(df),
        'year_365_rows': int(is_365.sum()),
        'days_rows': int(is_days.sum()),
        'year_360_rows': int(is_360.sum()),
        'exposure_std_min': df['exposure_standardised'].min(),
        'exposure_std_max': df['exposure_standardised'].max(),
        'total_written_premium': round(df['written_premium'].sum(), 2),
        'total_earned_premium': round(df['earned_premium'].sum(), 2),
    })

display(pd.DataFrame(std_summary))

## Roadmap: buildings claim cost model

Each step is tracked as a GitHub issue on the project Kanban board. Work through them in order, and reference the issue in the commit message (e.g. `Aggregate claims per row - closes #15`) so it closes when pushed.

**Target reminder** (from `docs/kaggle_overview.md`): predict the expected **ultimate claim cost** for each 2025 row: all claims with a `loss_date` in the row's period, net of excess and recoveries, excluding repudiated claims. Scored with **Tweedie D², power 1.5**.

### Done in this notebook (pending commit)

| Issue | Step | Cells |
| --- | --- | --- |
| #10 | Fix data loading paths for Databricks | 1-2 |
| #11 | Initial data preview and observations | 4-5 |
| #12 | Derive `policy_type` from the `policy_id` prefix | 6-9 |
| #13 | Add `earned_premium` | 10-11, 14 |
| #14 | Standardise exposure to a 365-day basis | 12-14 |

### Next steps

1. **#16 Develop claims to ultimate (chain ladder).** Claims are valued at 31 Dec 2024, so recent years, especially 2024, are under-reported (IBNR) and open claims may still develop (IBNER). Estimate development factors from reporting delay and incurred development, then add a developed column (e.g. `incurred_ultimate`) to `claims` **before** aggregating. Without this, the model would learn that recent years have lower claims than they really do.
2. **#15 Aggregate claims per policy row.** A `policy_id` spans several years and can have several rows per year, so match each claim to the policy row whose `period_start` to `period_end` contains its `loss_date`. Then, per row, calculate `claim_count`, `total_paid`, `total_incurred` and `total_recoveries`, and the target net of recoveries excluding repudiated claims. Check that totals reconcile to the raw claims.
3. **#17 Join to buildings policies.** Left-join the aggregates onto `policies_buildings` by `row_id`, filling rows with no claims with 0. Check the row count is unchanged (707,726) and that frequency and loss ratio (vs `earned_premium`) by year look reasonable. Optionally add `addresses`, `weather` and `customers` features.
4. **#18 Fit XGBoost (buildings).** Use `reg:tweedie` with `tweedie_variance_power = 1.5` to match the scoring metric, with `log(exposure_standardised)` as an offset. Validate on a time-based split (e.g. train 2021-2023, validate 2024), because policies recur across years. Exclude leaking columns (`renewal_decision`, IDs), clean `construction_type`, and compare against a constant-rate baseline.
5. **#19 Buildings-only submission.** Score `test_buildings`, merge onto `sample_submission` by `row_id`, and set motor (`M`) and contents (`C`) rows to 0. Check for 626,639 unique rows, no nulls and no negative predictions. Note that zero predictions for motor and contents will score poorly on those rows, so this is a first buildings-only check, not the final submission.

## Submission v001: buildings constant-rate baseline

A first end-to-end submission, made before the claims work (#15-#17) and the XGBoost model (#18) are done. It checks the file format and gives a leaderboard reference point.

* **Buildings rows**: a single burning-cost rate per year of exposure, multiplied by each test row's `exposure_standardised`.
  * Rate = total buildings claim cost / total buildings `exposure_standardised` (2021-2024 training data).
  * Claim cost = `incurred_to_date - recoveries`, excluding repudiated claims, with no chain ladder development (#16). This **understates** cost, especially for 2024.
* **Motor and contents rows**: 0, as agreed for the buildings-only submission (#19).
* A constant rate is the Tweedie D² baseline, so the buildings rows should score about 0. Any gain from later versions comes from the model.

The file is saved to `submissions/home_insurance_submission_v001.csv` in the repo, with header `row_id,prediction`. The cell checks for 626,639 unique `row_id`s that match `sample_submission`, no nulls and no negative predictions.

In [0]:
import os

sub_template = kw_data_typed['sample_submission']
test_b = kw_data_typed['test_buildings']
pol_b = kw_data_typed['policies_buildings']
claims = kw_data_typed['claims']

# Share of the submission that is buildings (by row_id prefix)
line_share = (sub_template['row_id'].str[0].map(policy_type_map)
              .value_counts().rename('rows').to_frame())
line_share['pct'] = (line_share['rows'] / len(sub_template) * 100).round(2)
display(line_share.reset_index())

# Buildings burning cost per year of exposure (no IBNR/IBNER development yet)
b_claims = claims[(claims['policy_type'] == 'buildings') & (claims['status'] != 'repudiated')]
b_claim_cost = (b_claims['incurred_to_date'] - b_claims['recoveries']).sum()
b_exposure = pol_b['exposure_standardised'].sum()
b_rate = b_claim_cost / b_exposure
print(f'Buildings claim cost: {b_claim_cost:,.0f} CU | exposure: {b_exposure:,.0f} years | rate: {b_rate:,.2f} CU per year')

# Predictions: buildings = rate * exposure, everything else = 0
b_pred = test_b[['row_id']].assign(prediction=test_b['exposure_standardised'] * b_rate)
submission = (sub_template[['row_id']]
              .merge(b_pred, on='row_id', how='left')
              .fillna({'prediction': 0.0}))
# Round prediction to 2 decimal places (stays float64)
submission['prediction'] = submission['prediction'].astype('float64').round(2)

# Checks
assert len(submission) == len(sub_template) == 626_639
assert submission['row_id'].is_unique
assert set(submission['row_id']) == set(sub_template['row_id'])
assert submission['prediction'].notna().all() and (submission['prediction'] >= 0).all()
assert (submission.loc[submission['row_id'].str[0] != 'B', 'prediction'] == 0).all()
assert submission['row_id'].isin(test_b['row_id']).sum() == len(test_b)

# Save
out_dir = '../submissions'
os.makedirs(out_dir, exist_ok=True)
out_path = f'{out_dir}/home_insurance_submission_v001.csv'
# Write prediction with exactly 2 decimal places, no index
submission.to_csv(out_path, index=False, float_format='%.2f')

print(f'Saved {len(submission):,} rows to {os.path.abspath(out_path)}')
display(submission.groupby(submission['row_id'].str[0].map(policy_type_map))['prediction']
        .agg(['count', 'sum', 'mean']).reset_index())

### Submission v001: results

**Buildings is 29.39% of the submission file** (184,155 of 626,639 rows, by `row_id` prefix):

| Line | Rows | % of submission | v001 prediction |
| --- | --- | --- | --- |
| Motor (`M`) | 254,986 | 40.69% | 0 |
| Contents (`C`) | 187,498 | 29.92% | 0 |
| Buildings (`B`) | 184,155 | 29.39% | rate × exposure |

So v001 gives a real prediction for under a third of rows. The other ~70.6% (motor and contents) are set to 0 and will score poorly. Motor is the largest line, so it should be the priority after buildings.

* **Buildings rate**: 248.2M CU claim cost / 590,560 exposure years = **420.28 CU per policy year**, giving a mean prediction of 342.89 CU per test row (many 2025 rows cover part of a year). Total buildings prediction is 63.1M CU.
* **Sense check**: the rate is above buildings earned premium per exposure year (191.0M / 590,560 = about 323 CU), a loss ratio of about 130%. This needs reviewing alongside the chain ladder work (#16) and the claims aggregation (#15).
* **File checks passed**: 626,639 unique `row_id`s matching `sample_submission`, header `row_id,prediction`, no nulls, no negatives, and non-buildings rows are exactly 0. Re-reading the saved CSV gave the same totals.
* **Saved to**: `submissions/home_insurance_submission_v001.csv` (repo root).

### Submission v001: parquet version and format check

The cell below saves the same submission as `submissions/home_insurance_submission_v001.parquet`. It then reads the parquet file back and compares its **structure** with `data/sample_submission.parquet`:

* Same parquet schema (column names, order and Arrow types)
* Same row count and the same `row_id`s **in the same order**
* Same pandas dtypes
* No nulls or negative predictions

The `prediction` values are expected to differ, because the sample is all zeros. The cell also confirms that the parquet and CSV versions contain identical data.

In [0]:
import pyarrow.parquet as pq

sample_path = '../data/sample_submission.parquet'
parquet_path = f'{out_dir}/home_insurance_submission_v001.parquet'

# Save with the same column order as the sample, no index
submission[sub_template.columns.tolist()].to_parquet(parquet_path, index=False)

# Read both back from disk
sample_df = pd.read_parquet(sample_path)
sub_pq = pd.read_parquet(parquet_path)
sub_csv = pd.read_csv(out_path, dtype={'row_id': str})

sample_schema = pq.read_schema(sample_path).remove_metadata()
sub_schema = pq.read_schema(parquet_path).remove_metadata()

checks = {
    'parquet schema matches sample': sub_schema.equals(sample_schema),
    'column names and order match': list(sub_pq.columns) == list(sample_df.columns),
    'pandas dtypes match': sub_pq.dtypes.equals(sample_df.dtypes),
    'row count matches (626,639)': len(sub_pq) == len(sample_df) == 626_639,
    'row_ids identical and in same order': sub_pq['row_id'].equals(sample_df['row_id']),
    'no nulls': bool(sub_pq.notna().all().all()),
    'no negative predictions': bool((sub_pq['prediction'] >= 0).all()),
    # CSV stores floats as text, so allow for float parsing round-off (max diff printed below)
    'parquet data matches CSV': bool(sub_pq['row_id'].equals(sub_csv['row_id'])
                                     and np.allclose(sub_pq['prediction'], sub_csv['prediction'], rtol=1e-12, atol=0)),
}
display(pd.DataFrame({'check': list(checks), 'passed': list(checks.values())}))
print('Sample schema:    ', sample_schema)
print('Submission schema:', sub_schema)
print('Max |parquet - CSV| prediction difference:', (sub_pq['prediction'] - sub_csv['prediction']).abs().max())
assert all(checks.values()), 'Submission does not match the sample format'
print(f'Saved {parquet_path}')

### Submission v001 parquet: check results

`submissions/home_insurance_submission_v001.parquet` **exactly matches the format of `sample_submission.parquet`**. Only the predictions differ, and they are ours:

| Check | Result |
| --- | --- |
| Parquet schema (`row_id: string`, `prediction: double`) | Identical to sample |
| Column names and order | Identical |
| pandas dtypes | Identical |
| Row count | 626,639 in both |
| `row_id`s and row order | Identical |
| Nulls / negative predictions | None |
| Parquet vs CSV | Same `row_id`s; predictions identical (both rounded to 2 decimal places) |

Predictions differ from the sample as intended (the sample is all 0). Predictions are float64 rounded to 2 decimal places (the CSV writes every value as e.g. `342.89` or `0.00`). Buildings rows range from 1.13 to 420.28 CU (total 63.1M CU), and motor and contents rows are 0. An independent re-read of both files with `pyarrow` confirmed the schema, row count and row order match.

In [0]:
display(kw_data['sample_submission'].head(20))

In [0]:
# Save sample_submission as CSV (no index) alongside the parquet in data/
sample_csv_path = '../data/sample_submission.csv'
kw_data['sample_submission'].to_csv(sample_csv_path, index=False, float_format='%.2f')

# Check: re-read and compare with the parquet original
sample_csv = pd.read_csv(sample_csv_path, dtype={'row_id': str})
assert list(sample_csv.columns) == ['row_id', 'prediction']
assert sample_csv['row_id'].equals(kw_data['sample_submission']['row_id'])
assert np.allclose(sample_csv['prediction'], kw_data['sample_submission']['prediction'])
print(f'Saved {len(sample_csv):,} rows to {os.path.abspath(sample_csv_path)}')
with open(sample_csv_path) as f:
    print(''.join([next(f) for _ in range(3)]))

### Actual vs expected: buildings v001

The test file (2025) has no actual claims, so actual vs expected (A/E) is measured on the **2021-2024 buildings training rows**:

* **Actual** per policy row = sum of `incurred_to_date - recoveries` for non-repudiated buildings claims whose `loss_date` falls in the row's `period_start` to `period_end` (matched on `policy_id`). This is undeveloped, the same basis as the v001 rate.
* **Expected** per policy row = the v001 rate (420.28 CU per year) × `exposure_standardised`.
* The 2025 test predictions (v001) are shown alongside for reference.

Most rows have no claims, so the histogram uses log-spaced bins for rows with cost > 0, and the zero-claim count is printed separately. The second chart shows A/E by policy year (year of `period_start`).

In [0]:
import matplotlib.pyplot as plt

# --- Actual claim cost per buildings training row (loss_date within the row's period) ---
pol = pol_b[['row_id', 'policy_id', 'period_start', 'period_end', 'exposure_standardised']].copy()
pol['period_start'] = pd.to_datetime(pol['period_start'])
pol['period_end'] = pd.to_datetime(pol['period_end'])

clm = b_claims[['claim_id', 'policy_id', 'loss_date', 'incurred_to_date', 'recoveries']].copy()
clm['loss_date'] = pd.to_datetime(clm['loss_date'])
clm['cost'] = clm['incurred_to_date'] - clm['recoveries']

matched = clm.merge(pol[['row_id', 'policy_id', 'period_start', 'period_end']], on='policy_id')
matched = matched[(matched['loss_date'] >= matched['period_start']) & (matched['loss_date'] <= matched['period_end'])]
assert matched['claim_id'].is_unique, 'a claim matched more than one policy row'
n_unmatched = len(clm) - len(matched)

actual = matched.groupby('row_id')['cost'].sum()
ave = pol.assign(actual=pol['row_id'].map(actual).fillna(0.0),
                 expected=pol['exposure_standardised'] * b_rate,
                 year=pol['period_start'].dt.year)

print(f'Claims matched to a policy row: {len(matched):,} of {len(clm):,} ({n_unmatched:,} unmatched, cost {clm.loc[~clm.claim_id.isin(matched.claim_id), "cost"].sum():,.0f} CU)')
print(f'Training rows with no claims: {(ave["actual"] == 0).sum():,} of {len(ave):,} ({(ave["actual"] == 0).mean():.1%})')
print(f'Rows with negative net cost (recoveries > incurred): {(ave["actual"] < 0).sum():,}')
print(f'Total actual: {ave["actual"].sum():,.0f} CU | total expected: {ave["expected"].sum():,.0f} CU | A/E: {ave["actual"].sum() / ave["expected"].sum():.3f}')

# --- A/E by policy year ---
by_year = ave.groupby('year').agg(rows=('row_id', 'size'), exposure=('exposure_standardised', 'sum'),
                                  actual=('actual', 'sum'), expected=('expected', 'sum'))
by_year['claim_rows'] = ave[ave['actual'] > 0].groupby('year').size()
by_year['a_over_e'] = (by_year['actual'] / by_year['expected']).round(3)
display(by_year.round(0).assign(a_over_e=by_year['a_over_e']).reset_index())

# --- Plots ---
test_pred = submission.loc[submission['row_id'].str[0] == 'B', 'prediction']
pos_actual = ave.loc[ave['actual'] > 0, 'actual']
bins = np.logspace(np.log10(0.5), np.log10(max(pos_actual.max(), 1000)), 60)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax = axes[0]
ax.hist(pos_actual, bins=bins, alpha=0.6, label=f'Actual, training rows with claims (n={len(pos_actual):,})')
ax.hist(ave['expected'], bins=bins, alpha=0.6, label=f'Expected, training rows (n={len(ave):,})')
ax.hist(test_pred, bins=bins, alpha=0.6, histtype='step', linewidth=2, color='black',
        label=f'Predicted v001, 2025 test rows (n={len(test_pred):,})')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Claim cost per policy row (CU, log scale)'); ax.set_ylabel('Rows (log scale)')
ax.set_title('Buildings: actual vs expected claim cost per row')
ax.legend(fontsize=8)

ax = axes[1]
x = np.arange(len(by_year))
ax.bar(x - 0.2, by_year['actual'] / 1e6, width=0.4, label='Actual')
ax.bar(x + 0.2, by_year['expected'] / 1e6, width=0.4, label='Expected')
for i, r in enumerate(by_year['a_over_e']):
    ax.text(i, max(by_year['actual'].iloc[i], by_year['expected'].iloc[i]) / 1e6, f'A/E {r:.2f}', ha='center', va='bottom')
ax.set_xticks(x); ax.set_xticklabels(by_year.index)
ax.set_ylabel('Claim cost (M CU)'); ax.set_title('Buildings: actual vs expected by policy year (undeveloped)')
ax.legend()
plt.tight_layout()
plt.show()

### Actual vs expected: findings

* **Reconciles**: 26,375 of 26,442 buildings claims matched to a policy row by `loss_date`. Matched actual (247.77M CU) plus the 67 unmatched claims (0.43M CU) equals the 248.20M CU used for the v001 rate. Overall A/E is 0.998 (1.000 by construction, less the unmatched claims).
* **Shape**: 96.4% of training rows have no claims, while expected cost is spread thinly over every row (0 to 420 CU). Rows with claims range from a few CU to six figures. A constant rate cannot separate risky rows from safe ones, which is why it scores about 0 on Tweedie D². The 2025 predictions follow the same shape as the training expected values.
* **Trend by year**: A/E rises from **0.91 (2021)** to 0.98, 1.03 and **1.07 (2024)**, even though 2024 is the least developed year. That points to claims inflation or a mix change, not only IBNR. Once claims are developed (#16), the 2024 A/E will be higher still, so a flat 2021-2024 average **understates 2025** cost. A trend term or a year feature is needed in #18.

## Submission v002: constant rate for every line (fixes #79)

**v001 is not a valid submission**: every `prediction` must be greater than zero, but v001 set all motor and contents rows (about 70.6% of the file) to 0.

v002 applies the v001 burning-cost approach to all three lines:

* Rate per line = total non-repudiated `incurred_to_date - recoveries` / total training `exposure_standardised` (2021-2024, undeveloped, same caveats as v001).
* Prediction = line rate × test `exposure_standardised`, rounded to 2 decimal places.
* Floored at 0.01 so no row can round down to 0 (the smallest test exposure is 0.0027 of a year).

The cell saves `submissions/home_insurance_submission_v002.csv` (CSV only, no parquet), then re-reads it and checks every prediction is > 0, there are no nulls, and the `row_id`s match `sample_submission` in the same order.

In [0]:
import os
import numpy as np

sub_template = kw_data_typed['sample_submission']
claims = kw_data_typed['claims']
valid_claims = claims[claims['status'] != 'repudiated']
out_dir = '../submissions'
os.makedirs(out_dir, exist_ok=True)

# Constant burning-cost rate per line, applied to each test row's exposure
rates, preds = [], []
for line in ['buildings', 'contents', 'motor']:
    c = valid_claims[valid_claims['policy_type'] == line]
    cost = (c['incurred_to_date'] - c['recoveries']).sum()
    exposure = kw_data_typed[f'policies_{line}']['exposure_standardised'].sum()
    rate = cost / exposure
    test = kw_data_typed[f'test_{line}']
    preds.append(test[['row_id']].assign(prediction=test['exposure_standardised'] * rate))
    rates.append({'line': line, 'claim_cost': round(cost, 2), 'exposure_years': round(exposure, 2),
                  'rate_per_year': round(rate, 2), 'test_rows': len(test)})
display(pd.DataFrame(rates))

# Keep sample_submission row order; round to 2dp and floor at 0.01 so every prediction > 0
submission_v002 = sub_template[['row_id']].merge(pd.concat(preds), on='row_id', how='left')
submission_v002['prediction'] = submission_v002['prediction'].astype('float64').round(2).clip(lower=0.01)

# Checks before saving
assert len(submission_v002) == len(sub_template) == 626_639
assert submission_v002['row_id'].is_unique
assert submission_v002['row_id'].equals(sub_template['row_id'])
assert submission_v002['prediction'].notna().all()
assert (submission_v002['prediction'] > 0).all()

# Save CSV only (no parquet output)
csv_v002 = f'{out_dir}/home_insurance_submission_v002.csv'
submission_v002.to_csv(csv_v002, index=False, float_format='%.2f')

# Re-read from disk and check independently
re_csv = pd.read_csv(csv_v002, dtype={'row_id': str})
checks = {
    'all predictions > 0': bool((re_csv['prediction'] > 0).all()),
    'no nulls': bool(re_csv.notna().all().all()),
    'row count 626,639': len(re_csv) == 626_639,
    'row_ids match sample in order': re_csv['row_id'].equals(sub_template['row_id']),
    'csv header is row_id,prediction': list(re_csv.columns) == ['row_id', 'prediction'],
    'csv matches in-memory predictions': bool(np.allclose(re_csv['prediction'], submission_v002['prediction'], atol=0.005)),
}
display(pd.DataFrame(checks.items(), columns=['check', 'passed']))
assert all(checks.values())

print(f'Saved {len(re_csv):,} rows to {os.path.abspath(csv_v002)}')
display(re_csv.groupby(re_csv['row_id'].str[0].map(policy_type_map))['prediction']
        .agg(['count', 'min', 'mean', 'max', 'sum']).reset_index())

# From XGBoost to an explainable GLM (issue #80)

**Why this section exists.** v002 (one constant rate per line) placed 30th of 34 on the Kaggle Part 1 leaderboard. That is expected, because a constant rate cannot separate high-risk from low-risk policies. The next step is a risk-differentiated technical price. A board and pricing committee will expect that price to be **explainable**: a rating structure they can read, challenge and sign off.

**The approach is two-stage, a common pattern in modern pricing teams:**

| Stage | Tool | Role | What the board sees |
| --- | --- | --- | --- |
| 1. Discovery | XGBoost (gradient-boosted trees, Tweedie loss) | Searches every candidate factor, non-linear shape and interaction with little manual specification. Used as a **benchmark and a map**, not as the production price. | Not shown directly |
| 2. Production | Tweedie GLM, log link | A multiplicative rating table: base rate × factor relativities. Each factor has a relativity, a confidence interval and a significance test. | Relativity tables, A/E charts, lift vs benchmark |

XGBoost answers *"what is in the data?"*. The GLM answers *"what do we choose to charge for, and can we defend it?"*. The gap in predictive power between the two (measured on a 2024 holdout) is the **price of explainability**. It tells the board how much accuracy, if any, is being given up.

**Common modelling basis (all three lines):**

* **Target**: claim cost per policy row = sum of `incurred_to_date - recoveries` over non-repudiated claims whose `loss_date` falls in the row's period. This is the Kaggle definition. It is undeveloped as at 31 Dec 2024 (see the IBNR note below).
* **Distribution**: Tweedie, variance power p = 1.5, log link. Tweedie with 1 < p < 2 is a compound Poisson-gamma: a point mass at zero (no claim) plus a continuous positive part (claim cost). It is the standard single-model alternative to separate frequency and severity models, and p = 1.5 is the Kaggle scoring metric.
* **Exposure**: `log(exposure_standardised)` as an offset (fixed coefficient of 1), so the model predicts cost per policy year and scales it by time on risk.
* **Validation**: train on 2021-2023, test on 2024. Policies renew across years, so a random split would leak the same policy into both sets. A time split also mirrors the real task, which is pricing a future year (2025).
* **Metric**: Tweedie D² (p = 1.5), the share of Tweedie deviance explained relative to a constant rate (0 = no better than v002; higher is better). This is the Kaggle metric.

## Step 1: Modelling dataset per line

The cell below builds one row-level modelling frame per line (training 2021-2024 and test 2025) by joining:

| Source | Join key | What it adds |
| --- | --- | --- |
| `policies_*` / `test_*` | (base table) | Rating factors on the policy, e.g. sum insured, excess, channel |
| `addresses` | `address_id` | Location and property: region, flood zone, elevation, year built, property type |
| `customers` | `customer_id` | Credit band, prior claims (declared at joining), occupation, marital status |
| `weather` | `address_id` + policy year | Annual hail, wind, freeze, rainfall and temperature for that address and year |
| `claims` | `policy_id` + `loss_date` in the row's period | Target: claim count and cost |

**Decisions (for review):**

* **Large-loss capping.** Individual claims go up to about 11.9M CU. Uncapped, a handful of catastrophic claims would dominate the fit and produce unstable relativities. Each claim is capped at its line's **99.5th percentile** for modelling. The excess above the cap is spread back as a flat **large-loss loading** (uncapped total / capped total), the standard actuarial treatment. Relativities are fitted on capped cost.
* **Excluded from features:**
  * Identifiers and dates.
  * `renewal_decision`: known only after the period, so it is target leakage.
  * `written_premium` / `earned_premium`: the insurer's current price. Using it would build a model of the old price rather than of risk, and could not feature in a defensible new rating structure.
  * `surname`, `date_of_birth`: personal data. Age is already on the policy.
* **Derived features**: `property_age` = policy year − `year_built`, and `customer_tenure` = policy year − `customer_since_year`. These are easier to band in a GLM than the raw years.
* **Construction type** is cleaned for case (`Brick`/`brick`/`BRICK` → `brick`). The single-letter codes `T` and `W` are kept as separate levels (`code_t`, `code_w`), because the data dictionary does not define them (`T` is probably timber). **They need confirming with the data owner.**

Checks: row counts are unchanged by the joins, each claim matches at most one policy row, and matched plus unmatched claim cost reconciles to the raw claims file.

In [0]:
import numpy as np
import pandas as pd

LINES = ['buildings', 'contents', 'motor']
CAP_Q = 0.995  # per-claim large-loss cap: 99.5th percentile of claim cost, per line

addresses = kw_data_typed['addresses']
customers = kw_data_typed['customers'].drop(columns=['surname', 'date_of_birth'])
weather = kw_data_typed['weather'].rename(columns={'year': 'policy_year'})

# Non-repudiated claims, cost net of recoveries, capped per line
clm = kw_data_typed['claims'].query("status != 'repudiated'").copy()
clm['loss_date'] = pd.to_datetime(clm['loss_date'])
clm['cost'] = clm['incurred_to_date'] - clm['recoveries']
caps = clm.groupby('policy_type')['cost'].quantile(CAP_Q)
clm['cost_capped'] = np.minimum(clm['cost'], clm['policy_type'].map(caps))

EXCLUDE = {'row_id', 'policy_id', 'customer_id', 'address_id', 'period_start', 'period_end', 'exposure',
           'exposure_standardised', 'written_premium', 'earned_premium', 'renewal_decision', 'policy_type',
           'policy_year', 'claim_count', 'cost', 'cost_capped', 'compulsory_excess'}

def build_frame(line, kind):
    df = kw_data_typed[f'{kind}_{line}'].copy()
    df['period_start'] = pd.to_datetime(df['period_start'])
    df['period_end'] = pd.to_datetime(df['period_end'])
    df['policy_year'] = df['period_start'].dt.year
    n0 = len(df)
    df = (df.merge(addresses, on='address_id', how='left', validate='m:1')
            .merge(customers, on='customer_id', how='left', validate='m:1')
            .merge(weather, on=['address_id', 'policy_year'], how='left', validate='m:1'))
    assert len(df) == n0, f'{kind}_{line}: join changed row count'
    df['property_age'] = df['policy_year'] - df['year_built']
    df['customer_tenure'] = df['policy_year'] - df['customer_since_year']
    if 'construction_type' in df:
        df['construction_type'] = df['construction_type'].str.lower().replace({'t': 'code_t', 'w': 'code_w'})
    recon = None
    if kind == 'policies':
        c = clm[clm['policy_type'] == line]
        m = c.merge(df[['row_id', 'policy_id', 'period_start', 'period_end']], on='policy_id')
        m = m[(m['loss_date'] >= m['period_start']) & (m['loss_date'] <= m['period_end'])]
        assert m['claim_id'].is_unique, f'{line}: a claim matched more than one policy row'
        agg = m.groupby('row_id').agg(claim_count=('claim_id', 'size'), cost=('cost', 'sum'),
                                      cost_capped=('cost_capped', 'sum'))
        df = df.merge(agg, on='row_id', how='left').fillna({'claim_count': 0, 'cost': 0.0, 'cost_capped': 0.0})
        unmatched = c[~c['claim_id'].isin(m['claim_id'])]
        recon = {'line': line, 'claims': len(c), 'matched': len(m), 'unmatched': len(unmatched),
                 'reconciles': bool(np.isclose(df['cost'].sum() + unmatched['cost'].sum(), c['cost'].sum())),
                 'cap_CU': round(caps[line], 0),
                 'claims_capped': int((c['cost'] > caps[line]).sum()),
                 'large_loss_loading': round(m['cost'].sum() / m['cost_capped'].sum(), 4)}
    return df, recon

frames, recons = {}, []
for line in LINES:
    train, recon = build_frame(line, 'policies')
    test, _ = build_frame(line, 'test')
    feats = [c for c in train.columns if c not in EXCLUDE]
    # Consistent dtypes: bool -> int, text -> category with training levels (unseen test levels -> missing)
    for col in feats:
        if train[col].dtype == bool or train[col].dtype == object and set(train[col].dropna().unique()) <= {True, False}:
            train[col] = train[col].astype('float'); test[col] = test[col].astype('float')
        elif train[col].dtype == object or col == 'branch_id':
            cats = sorted(train[col].dropna().astype(str).unique())
            train[col] = pd.Categorical(train[col].astype('string'), categories=cats)
            test[col] = pd.Categorical(test[col].astype('string'), categories=cats)
    frames[line] = {'train': train, 'test': test, 'features': feats}
    recons.append({**recon, 'train_rows': len(train), 'test_rows': len(test), 'n_features': len(feats)})

display(pd.DataFrame(recons))
feature_types = pd.DataFrame([{'line': l, 'feature': f, 'type': 'categorical' if str(frames[l]['train'][f].dtype) == 'category' else 'numeric',
                               'levels / distinct': frames[l]['train'][f].nunique(), 'missing %': round(frames[l]['train'][f].isna().mean() * 100, 1)}
                              for l in LINES for f in frames[l]['features']])
display(feature_types)

# Free memory (serverless limit): raw, untyped copies of the policy and test files are not used below
import gc
for k in [f'{kind}_{l}' for kind in ('policies', 'test') for l in LINES]:
    kw_data.pop(k, None)
_ = gc.collect()

## Step 2: Experience by policy year, and a leakage screen

Before modelling, two checks that affect how the results should be read:

1. **Claim frequency and capped loss cost per exposure year, by policy year and line.** If 2024 is noticeably lower than 2021-2023, that points to incomplete development (IBNR/IBNER at the 31 Dec 2024 valuation), not better risk. The holdout year (2024) is affected, so holdout D² is reported both as is and **level-adjusted** (predictions rescaled to the holdout's actual total). The level-adjusted figure isolates **risk ranking**, which is what relativities are about. The overall level is a separate base-rate and development question (#16).
2. **Leakage screen.** A feature is leakage if it is recorded *because of* a claim. For each line, the screen compares the average of each numeric feature for rows with and without a claim, and ranks features by the standardised difference. A very large gap is a warning sign that needs a business explanation. `customer_service_contacts` is the main suspect: policyholders who claim typically call the insurer, so the field may be partly a consequence of the claim rather than a cause.

In [0]:
import matplotlib.pyplot as plt

year_rows = []
for line in LINES:
    t = frames[line]['train']
    g = t.groupby('policy_year').agg(exposure=('exposure_standardised', 'sum'), claims=('claim_count', 'sum'),
                                     cost_capped=('cost_capped', 'sum'))
    g['frequency'] = g['claims'] / g['exposure']
    g['loss_cost_capped'] = g['cost_capped'] / g['exposure']
    year_rows.append(g.assign(line=line).reset_index())
by_year = pd.concat(year_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for line, g in by_year.groupby('line'):
    base = g.set_index('policy_year').loc[2021]
    axes[0].plot(g['policy_year'], g['frequency'] / base['frequency'], marker='o', label=line)
    axes[1].plot(g['policy_year'], g['loss_cost_capped'] / base['loss_cost_capped'], marker='o', label=line)
for ax, t in zip(axes, ['Claim frequency (index, 2021 = 1)', 'Capped loss cost per exposure year (index, 2021 = 1)']):
    ax.axhline(1, color='grey', lw=0.8, ls='--'); ax.set_title(t); ax.set_xticks([2021, 2022, 2023, 2024]); ax.legend()
plt.tight_layout(); plt.show()
display(by_year[['line', 'policy_year', 'exposure', 'claims', 'frequency', 'loss_cost_capped']].round(4))

# Leakage screen: standardised mean difference, rows with a claim vs without
screen = []
for line in LINES:
    t = frames[line]['train']
    has = t['claim_count'] > 0
    for f in frames[line]['features']:
        if str(t[f].dtype) == 'category':
            continue
        x = t[f].astype(float)
        smd = (x[has].mean() - x[~has].mean()) / x.std()
        screen.append({'line': line, 'feature': f, 'mean_with_claim': x[has].mean(), 'mean_no_claim': x[~has].mean(),
                       'std_mean_diff': smd})
screen = pd.DataFrame(screen)
screen['abs_smd'] = screen['std_mean_diff'].abs()
display(screen.sort_values('abs_smd', ascending=False).groupby('line').head(6).round(3))

# Main suspect: customer_service_contacts. Compare distributions in training (by claim status) and 2025 test
csc = pd.DataFrame([{'line': l,
                     'train_no_claim': frames[l]['train'].loc[frames[l]['train']['claim_count'] == 0, 'customer_service_contacts'].mean(),
                     'train_with_claim': frames[l]['train'].loc[frames[l]['train']['claim_count'] > 0, 'customer_service_contacts'].mean(),
                     'train_all': frames[l]['train']['customer_service_contacts'].mean(),
                     'test_2025': frames[l]['test']['customer_service_contacts'].mean()} for l in LINES])
display(csc.round(3))

### Findings: experience by year and leakage

**Development (IBNR).** Motor 2024 frequency is 0.1295 claims per policy year, against 0.140-0.149 in 2021-2023, and capped loss cost is down 17% on 2021. That pattern is typical of claims not yet reported at the 31 Dec 2024 valuation, rather than better risk. Buildings and contents frequency is flat, while buildings loss cost rises steadily (368 → 428 CU, about 5% a year), which looks like claims inflation. Consequences:

* Motor holdout D² *as is* is distorted by the under-reported 2024 level. The **level-adjusted** D² is the better measure of risk ranking.
* Policy year is **not** used as a rating factor. Trend and development belong in the base rate (#16), not in relativities.

**Leakage: `customer_service_contacts` is excluded from all models.** The screen flags it as by far the strongest signal on every line (rows with a claim average about 3.0-3.3 contacts, against about 0.4 without). The 2025 test file confirms the mechanism: its average (0.40) matches training rows **without** a claim, not the training average (0.50-0.69). So the field counts contacts *caused by* claims during the period, and those contacts are not present in the data the model is asked to predict. A model using it would look excellent in training and fail on 2025. It could not be used at the point of quote either. *Please confirm with the data owner.*

The other features show modest differences with a clear risk rationale (e.g. motor NCB years lower for claimants, buildings older properties and higher flood zones), so they stay in as candidates.

## Step 3: XGBoost benchmark per line

**Specification and rationale:**

| Setting | Value | Why |
| --- | --- | --- |
| Objective | `reg:tweedie`, variance power 1.5 | Same loss as the Kaggle metric, and the same distribution as the planned GLM, so the two are directly comparable |
| Offset | `base_margin = log(exposure_standardised)` | Equivalent to a GLM exposure offset: the trees model cost per policy year |
| Target | Capped claim cost (99.5th percentile per claim) | Stable relativities; large losses handled by a flat loading |
| Offset level | `log(base rate)` is added to the offset (base rate = 2021-2023 capped cost / exposure) | The trees start at the right overall level, so they only learn **relativities** around it, as in a GLM |
| Trees | depth 5, learning rate 0.05, `min_child_weight` 1,000, row and column subsampling 0.8 | Shallow trees with large leaves: enough to capture two- and three-way interactions, while each leaf rests on a credible volume of data |
| Balance correction | Predictions rescaled so that 2021-2023 predicted total = actual total | Boosted Tweedie trees under-shoot the overall level: Newton steps on the log scale are damped by rows with large claims. A single scalar restores the balance property a GLM with an intercept has by construction, and does not change any relativity |
| Rounds | up to 1,500, early stopping after 50 rounds without holdout improvement | Number of trees chosen on 2024 out-of-time performance |
| Categoricals | native XGBoost categorical splits | No arbitrary one-hot or ordinal coding imposed before discovery |
| Missing values | native XGBoost handling | The model learns which branch missing values follow, which reveals whether "missing" carries risk |

Training is on 2021-2023 and the holdout is 2024. Each run is logged to MLflow (parameters, holdout metrics and the model) for reproducibility. The 2021-2023 model is the one interpreted below. A refit on 2021-2024 is deferred to the submission step.

In [0]:
import time
import xgboost as xgb
import mlflow
from sklearn.metrics import d2_tweedie_score

LEAKAGE = {'customer_service_contacts'}
FEATS = {l: [f for f in frames[l]['features'] if f not in LEAKAGE] for l in LINES}
TARGET = 'cost_capped'

XGB_PARAMS = {'objective': 'reg:tweedie', 'tweedie_variance_power': 1.5, 'eval_metric': 'tweedie-nloglik@1.5',
              'eta': 0.05, 'max_depth': 5, 'min_child_weight': 1000, 'subsample': 0.8, 'colsample_bytree': 0.8,
              'lambda': 1.0, 'tree_method': 'hist', 'max_cat_to_onehot': 1, 'nthread': 4, 'seed': 42}

def split(line):
    t = frames[line]['train']
    return t[t['policy_year'] <= 2023], t[t['policy_year'] == 2024]

# Base rate per policy year (2021-2023 capped cost / exposure). It goes into the offset so the trees only
# have to learn relativities around it, exactly as in a GLM: prediction = exposure x base rate x relativities
BASE_RATE = {l: split(l)[0][TARGET].sum() / split(l)[0]['exposure_standardised'].sum() for l in LINES}

def dmatrix(df, line, with_label=True):
    return xgb.DMatrix(df[FEATS[line]], label=df[TARGET] if with_label else None,
                       base_margin=np.log(df['exposure_standardised'] * BASE_RATE[line]), enable_categorical=True)

def d2_scores(y, pred):
    return d2_tweedie_score(y, pred, power=1.5), d2_tweedie_score(y, pred * y.sum() / pred.sum(), power=1.5)

mlflow.set_experiment('/Users/alan.mcdonagh@rgare.com/iaa_hackathon_2026_pricing')
xgb_models, holdout, xgb_results = {}, {}, []
for line in LINES:
    tr, ho = split(line)
    dtr, dho = dmatrix(tr, line), dmatrix(ho, line)
    t0 = time.time()
    with mlflow.start_run(run_name=f'xgb_tweedie_{line}'):
        bst = xgb.train(XGB_PARAMS, dtr, num_boost_round=1500, evals=[(dho, 'holdout_2024')],
                        early_stopping_rounds=50, verbose_eval=False)
        it = (0, bst.best_iteration + 1)
        # Balance correction: boosted Tweedie trees do not reproduce the training total exactly (Newton steps on
        # the log scale are damped for rows with large claims), so rescale so that 2021-2023 predicted = actual.
        cal = tr[TARGET].sum() / bst.predict(dtr, iteration_range=it).sum()
        pred = bst.predict(dho, iteration_range=it) * cal
        d2_raw, d2_adj = d2_scores(ho[TARGET], pred)
        res = {'line': line, 'base_rate_CU': BASE_RATE[line], 'balance_factor': cal, 'train_rows': len(tr), 'holdout_rows': len(ho), 'best_trees': bst.best_iteration + 1,
               'holdout_D2': d2_raw, 'holdout_D2_level_adj': d2_adj,
               'pred_over_actual_2024': pred.sum() / ho[TARGET].sum(), 'minutes': (time.time() - t0) / 60}
        mlflow.log_params({**XGB_PARAMS, 'line': line, 'target': TARGET, 'cap_quantile': CAP_Q,
                           'train_years': '2021-2023', 'holdout_year': 2024, 'n_features': len(FEATS[line])})
        mlflow.log_metrics({k: float(v) for k, v in res.items() if k not in ('line',)})
        mlflow.xgboost.log_model(bst, name='model')
    xgb_models[line] = {'booster': bst, 'iteration_range': it, 'balance_factor': cal}
    holdout[line] = ho[['row_id', 'exposure_standardised', TARGET, 'cost', 'claim_count']].assign(pred_xgb=pred)
    xgb_results.append(res)
    print(f"{line}: {res['best_trees']} trees, holdout D2 {d2_raw:.4f} (level-adjusted {d2_adj:.4f}), {res['minutes']:.1f} min")

xgb_results = pd.DataFrame(xgb_results)
display(xgb_results.round(4))

## Step 4: Which features matter? Importance is not significance

XGBoost has no coefficients, standard errors or p-values, so "significance" in the GLM sense does not exist for it. It offers three different measures of how much a feature is used. An actuarial reader should treat them as follows:

| Measure | What it is | Strength | Weakness |
| --- | --- | --- | --- |
| **Gain** (share of total loss reduction from splits on the feature) | In-sample: how much training deviance the feature's splits removed | Free, built into the model | Biased towards continuous and high-cardinality features (more split points to choose from). In-sample only. |
| **Mean \|SHAP\|** (average absolute contribution to log prediction) | In-sample: on average, how far the feature moves a policy's predicted log cost away from the base rate | Consistent, additive attribution on the **log scale**, which is the same scale as GLM coefficients. exp(SHAP) is a multiplicative relativity. | Measures *use*, not *predictive value*. A feature can move predictions a lot and still be fitting noise. |
| **Permutation importance** on the 2024 holdout | Out-of-time: how much worse holdout Tweedie deviance gets when the feature's values are randomly shuffled, breaking its link with the target | The closest analogue to significance: it measures **genuine out-of-sample predictive value**. Repeated shuffles give a spread, so we can check whether the effect is clearly above zero. | Correlated features share credit, so shuffling one lets its partner compensate, and both can look weaker than they are |

**How the results are used to pick GLM candidates:**

* A feature is a **strong candidate** if its permutation importance is clearly positive (the mean minus 2 standard errors over 5 shuffles is above 0) **and** it ranks highly on mean |SHAP|.
* A feature with high gain or SHAP but **no** permutation value is fitting noise in training. Leave it out of the GLM.
* Permutation importance is shown as a **% of the deviance explained by the model on the holdout**. For example, 20% means shuffling that feature alone loses a fifth of the model's improvement over a flat rate.
* Formal statistical significance (likelihood-ratio tests and p-values) is then established properly in the **GLM** (Step 6), where it is well defined.

In [0]:
from sklearn.metrics import mean_tweedie_deviance

rng = np.random.default_rng(42)
N_SHAP, N_PERM, REPEATS = 20_000, 100_000, 5

def xgb_predict(line, X, exposure):
    m = xgb_models[line]
    d = xgb.DMatrix(X[FEATS[line]], base_margin=np.log(exposure * BASE_RATE[line]), enable_categorical=True)
    return m['booster'].predict(d, iteration_range=m['iteration_range']) * m['balance_factor']

def shap_values(line, X):
    """Exact TreeSHAP from XGBoost on the log (margin) scale; drops the bias column."""
    m = xgb_models[line]
    d = xgb.DMatrix(X[FEATS[line]], enable_categorical=True)
    contrib = m['booster'].predict(d, pred_contribs=True, iteration_range=m['iteration_range'])
    return pd.DataFrame(contrib[:, :-1], columns=FEATS[line], index=X.index)

shap_samples, imp_all = {}, []
for line in LINES:
    tr, ho = split(line)
    bst = xgb_models[line]['booster']
    # 1. Gain
    gain = pd.Series(bst.get_score(importance_type='total_gain')).reindex(FEATS[line]).fillna(0)
    # 2. Mean |SHAP| on a random training sample
    s = tr.sample(N_SHAP, random_state=42)
    sv = shap_values(line, s)
    shap_samples[line] = (s, sv)
    mean_abs_shap = sv.abs().mean()
    # 3. Permutation importance on a 2024 holdout sample (level-adjusted predictions, Tweedie deviance p=1.5)
    h = ho.sample(min(N_PERM, len(ho)), random_state=42).reset_index(drop=True)
    y, e = h[TARGET].to_numpy(), h['exposure_standardised'].to_numpy()
    p0 = xgb_predict(line, h, e); k = y.sum() / p0.sum()
    dev_model = mean_tweedie_deviance(y, p0 * k, power=1.5)
    dev_null = mean_tweedie_deviance(y, e * y.sum() / e.sum(), power=1.5)
    explained = dev_null - dev_model
    perm = {}
    for f in FEATS[line]:
        Xp, losses = h.copy(), []
        for _ in range(REPEATS):
            Xp[f] = h[f].iloc[rng.permutation(len(h))].set_axis(h.index)
            losses.append((mean_tweedie_deviance(y, xgb_predict(line, Xp, e) * k, power=1.5) - dev_model) / explained * 100)
        perm[f] = (np.mean(losses), np.std(losses, ddof=1) / np.sqrt(REPEATS))
    imp = pd.DataFrame({'feature': FEATS[line], 'gain_share_%': (gain / gain.sum() * 100).values,
                        'mean_abs_shap': mean_abs_shap.values,
                        'perm_%_of_explained': [perm[f][0] for f in FEATS[line]],
                        'perm_se': [perm[f][1] for f in FEATS[line]]})
    imp['shap_share_%'] = imp['mean_abs_shap'] / imp['mean_abs_shap'].sum() * 100
    imp['predictive'] = (imp['perm_%_of_explained'] - 2 * imp['perm_se']) > 0
    imp['line'] = line
    imp_all.append(imp.sort_values('perm_%_of_explained', ascending=False))
    print(f'{line}: holdout sample deviance explained {explained / dev_null:.2%}; '
          f"{imp['predictive'].sum()} of {len(imp)} features clearly predictive out-of-time")
imp_all = pd.concat(imp_all, ignore_index=True)

# Chart: top 20 per line, ranked by out-of-time permutation importance, with the in-sample measures alongside
for line in LINES:
    d = imp_all[imp_all['line'] == line].head(20).iloc[::-1]
    fig, axes = plt.subplots(1, 3, figsize=(16, 7), sharey=True)
    colours = np.where(d['predictive'], 'tab:blue', 'lightgrey')
    axes[0].barh(d['feature'], d['perm_%_of_explained'], xerr=2 * d['perm_se'], color=colours)
    axes[0].set_title('Permutation (2024 holdout)\n% of explained deviance lost, ±2 SE')
    axes[1].barh(d['feature'], d['shap_share_%'], color='tab:orange'); axes[1].set_title('Mean |SHAP| share % (in-sample)')
    axes[2].barh(d['feature'], d['gain_share_%'], color='tab:green'); axes[2].set_title('Gain share % (in-sample)')
    fig.suptitle(f'{line.title()}: top 20 features (blue = clearly predictive out-of-time)', fontsize=14)
    plt.tight_layout(); plt.show()

display(imp_all.groupby('line').head(15).round(3))

## Step 5: Shape of each effect: SHAP relativity curves

For each top feature, the charts below show **exp(SHAP)**: the multiplicative factor the XGBoost model applies to a policy's predicted cost because of that feature's value, holding the effect of the other features fixed. This is read exactly like a GLM relativity (1.20 = 20% above average risk). Two curves are overlaid:

* **Blue: XGBoost (SHAP) relativity**, averaged within bins of the feature. This is the *adjusted* effect, after allowing for every other factor in the model.
* **Grey: one-way observed relativity** = (capped loss cost per exposure year in the bin) / (overall loss cost per exposure year), on all 2021-2023 data. This is the *unadjusted* effect a traditional one-way analysis would show.

**How to read the gap:** where grey is steeper than blue, part of the one-way effect is really caused by a correlated factor (e.g. an elevation effect that is partly flood zone). That is exactly what a multivariate GLM corrects for, and what a one-way rating table would double-count. The blue shape tells us **how to bring the factor into the GLM**:

| SHAP shape | GLM treatment |
| --- | --- |
| Straight line on a log scale | Single linear term (or linear in log(x)) |
| Monotone curve that flattens | Log transform, or a few bands |
| U-shape, or steps | Bands (piecewise-constant factor), or a spline |
| Flat | Drop the factor |

The light dots are individual policies (a sample of 4,000). Vertical spread at the same x value means the effect depends on other factors, i.e. **interactions**, which are quantified in Step 5b.

In [0]:
TOP_N = 9

def one_way(tr, f, bins=None):
    g = tr.groupby(bins if bins is not None else tr[f], observed=True).agg(exp=('exposure_standardised', 'sum'), cost=(TARGET, 'sum'))
    return (g['cost'] / g['exp']) / (tr[TARGET].sum() / tr['exposure_standardised'].sum()), g['exp']

for line in LINES:
    tr, _ = split(line)
    s, sv = shap_samples[line]
    top = imp_all[(imp_all['line'] == line) & imp_all['predictive']].head(TOP_N)['feature'].tolist()
    fig, axes = plt.subplots(3, 3, figsize=(17, 13)); axes = axes.ravel()
    for ax, f in zip(axes, top):
        rel = np.exp(sv[f])
        if str(tr[f].dtype) == 'category' or tr[f].nunique() <= 12:
            lv = s[f].astype(str)
            shap_rel = rel.groupby(lv).mean()
            ow, ex = one_way(tr, f)
            ow.index = ow.index.astype(str); ex.index = ex.index.astype(str)
            levels = ex.sort_values(ascending=False).index[:15]
            levels = [l for l in levels if l in shap_rel.index]
            order = sorted(levels, key=lambda l: shap_rel[l])
            x = np.arange(len(order))
            ax.bar(x - 0.2, shap_rel[order], 0.4, label='XGBoost (SHAP)', color='tab:blue')
            ax.bar(x + 0.2, ow.reindex(order), 0.4, label='One-way observed', color='lightgrey')
            ax.set_xticks(x); ax.set_xticklabels(order, rotation=60, ha='right', fontsize=8)
        else:
            q = np.unique(np.nanquantile(tr[f], np.linspace(0, 1, 21)))
            ax.scatter(s[f].iloc[:4000], rel.iloc[:4000], s=2, alpha=0.15, color='tab:blue')
            sb = pd.cut(s[f], q, include_lowest=True); mids = sb.cat.categories.mid
            ax.plot(mids, rel.groupby(sb, observed=False).mean().values, color='tab:blue', lw=2, marker='o', ms=3, label='XGBoost (SHAP)')
            ow, _ = one_way(tr, f, pd.cut(tr[f], q, include_lowest=True))
            ax.plot(ow.index.categories.mid if hasattr(ow.index, 'categories') else [i.mid for i in ow.index], ow.values,
                    color='grey', lw=2, ls='--', marker='s', ms=3, label='One-way observed')
            ax.set_ylim(0, max(2.5, np.nanpercentile(rel, 99.5)))
        pi = imp_all[(imp_all['line'] == line) & (imp_all['feature'] == f)]['perm_%_of_explained'].iloc[0]
        ax.axhline(1, color='black', lw=0.7); ax.set_title(f'{f}  (permutation {pi:.1f}%)', fontsize=11)
        ax.set_ylabel('relativity')
    axes[0].legend(fontsize=8)
    for ax in axes[len(top):]: ax.axis('off')
    fig.suptitle(f'{line.title()}: relativity curves for the top {len(top)} out-of-time predictive features', fontsize=15)
    plt.tight_layout(); plt.show()

# All clearly predictive features per line (GLM candidate list)
display(imp_all[imp_all['predictive']].groupby('line')['feature'].apply(lambda x: ', '.join(x)).reset_index())

In [0]:
# Numeric version of the curves above (decile bins), used to choose GLM transforms and band edges
rel_tables = []
for line in LINES:
    s, sv = shap_samples[line]
    top = imp_all[(imp_all['line'] == line) & imp_all['predictive']].head(12)['feature'].tolist()
    for f in top:
        if str(s[f].dtype) == 'category':
            continue
        if s[f].nunique() <= 12:
            b = s[f]
        else:
            b = pd.qcut(s[f], 10, duplicates='drop')
        t = np.exp(sv[f]).groupby(b, observed=True).agg(['mean', 'size'])
        rel_tables.append(pd.DataFrame({'line': line, 'feature': f, 'bin': t.index.astype(str),
                                        'shap_relativity': t['mean'].round(3).values, 'n_sample': t['size'].values}))
rel_tables = pd.concat(rel_tables, ignore_index=True)
display(rel_tables)

### Step 5b: Interactions

A GLM is additive on the log scale (the factors multiply) unless interaction terms are added explicitly. XGBoost finds interactions automatically. **SHAP interaction values** split each prediction into main effects plus a contribution for each pair of features. The mean absolute pairwise interaction, as a share of the mean absolute main effects, shows whether any pair is strong enough to justify an interaction term (or a combined factor) in the GLM.

A rule of thumb: pairs below about 5% of the main-effect total add complexity without much benefit and are better left out. Larger pairs are tested formally in the GLM (likelihood-ratio test) before being accepted.

In [0]:
N_INT = 3_000
interactions = []
for line in LINES:
    s = shap_samples[line][0].iloc[:N_INT]
    m = xgb_models[line]
    d = xgb.DMatrix(s[FEATS[line]], enable_categorical=True)
    iv = m['booster'].predict(d, pred_interactions=True, iteration_range=m['iteration_range'])[:, :-1, :-1]
    mean_abs = np.abs(iv).mean(axis=0)
    main_total = np.diag(mean_abs).sum()
    F = FEATS[line]
    for i in range(len(F)):
        for j in range(i + 1, len(F)):
            interactions.append({'line': line, 'feature_1': F[i], 'feature_2': F[j],
                                 'mean_abs_interaction': 2 * mean_abs[i, j],
                                 'pct_of_main_effects': 2 * mean_abs[i, j] / main_total * 100})
interactions = pd.DataFrame(interactions).sort_values('mean_abs_interaction', ascending=False)
top_int = interactions.groupby('line').head(8)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, line in zip(axes, LINES):
    d = top_int[top_int['line'] == line].iloc[::-1]
    ax.barh(d['feature_1'] + ' × ' + d['feature_2'], d['pct_of_main_effects'], color='tab:purple')
    ax.axvline(5, color='red', ls='--', lw=1); ax.set_title(f'{line.title()}: top pairwise interactions')
    ax.set_xlabel('% of total main-effect SHAP')
plt.tight_layout(); plt.show()
display(top_int.round(4))

### Findings: what the XGBoost models tell us

**Benchmark performance (2024 holdout, capped cost).** D²: buildings 0.101, contents 0.077, motor 0.080 (0.082 level-adjusted). A constant rate (v002) scores 0 by definition. Row-level D² of 0.05-0.15 is typical for personal lines: most policies have no claim in a year, so most of the variation is irreducible chance. What matters is the gain over a flat rate.

**The key risk drivers are few, and they make actuarial sense.** Ranked by out-of-time permutation importance (% of the model's explained deviance lost when the feature is shuffled):

| Line | Main drivers (permutation %) | Interpretation |
| --- | --- | --- |
| Buildings | flood zone (39%), elevation (32%), sum insured (26%), distance to river (25%), then property age (5%) and region (4%) | A **flood / water peril** story plus size of risk. Flood zones 4-5 are about 1.8× and 2.7× the average. Elevation ≤ 1.8 m is about 1.4×. Within 0.45 km of a river is about 1.13×. Relativity rises steadily with sum insured (0.63 to 1.28 across deciles). |
| Contents | sum insured (48%), flood zone (40%), elevation (23%), then voluntary excess, region and prior claims (5-6% each) | Size of risk plus the same flood geography. Zone 5 is about 3×. Excess of 500+ reduces cost by about 13%. |
| Motor | NCB years (51%), cover type (26%), annual mileage (15%), driver age (11%), make/model (9%), then excess, years licensed, region and prior claims (6-7% each) | The classic motor tariff. 0-1 years NCB is about 1.5×, against about 0.76× for 10+. Drivers aged 17-22 are about 1.3×. Mileage over 20,000 is about 1.27×. |

Factor sums exceed 100% because correlated factors (e.g. flood zone, elevation, river distance) each lose a share of a common signal when shuffled.

**In-sample importance can mislead, which is why permutation importance is used for decisions.** `previous_insurer` (40 levels) and `branch_id` (25 levels) carry 15-19% of buildings **gain**, yet `branch_id` has **no** clear out-of-time value and `previous_insurer` only about 3%. Both are high-cardinality fields that trees can use to memorise training noise. `make_model` (300 levels) has 42% of motor gain but 9% permutation value. It is real but over-weighted in-sample, and needs **credibility-weighted grouping** in a GLM rather than 300 raw levels.

**Adjusted vs one-way.** In the charts, the one-way (grey) effect of elevation and river distance is steeper than the SHAP (blue) effect, because low-lying, riverside properties also tend to be in high flood zones. A one-way rating table would **double-count** flood risk. A multivariate GLM avoids this.

**Interactions are modest.** The largest pair is flood zone × elevation (about 5% of main effects for buildings and contents): low-lying properties in high flood zones are worse than the two factors multiplied together. Every other pair is under 5%. This is good news for explainability: a **main-effects GLM plus at most one interaction per line** should capture most of the XGBoost signal.

## Step 6: Designing the explainable GLM

**Model form (all lines):**

expected cost = exposure × base rate × relativity(factor 1) × relativity(factor 2) × ...

This is a Tweedie GLM (p = 1.5) with log link and a `log(exposure)` offset, fitted on capped cost for 2021-2023. Each factor's base level is its **largest-exposure level** (relativity 1.00), so the base rate describes the most common risk.

**Factor selection rules (applied to the XGBoost evidence):**

1. Include a factor only if it is **clearly predictive out-of-time** (permutation test, Step 4) **and** has a plausible causal or underwriting rationale.
2. Use the **SHAP relativity shape** to choose the treatment: bands where the curve steps or flattens, `log(x)` where it is smooth and monotone.
3. Exclude predictive fields that a board, regulator or customer could not reasonably accept as rating factors, or that are proxies for the insurer's own processes: `previous_insurer`, `branch_id`, `campaign_code`, `marketing_segment`, `document_preference`, `days_quote_to_inception`, `discount_applied`, `vehicle_colour`. Their combined predictive contribution is small (mostly 1-4% each), and their inclusion would be hard to defend. **This is a business decision for the pricing committee**, and it is flagged here rather than made silently.
4. Add an interaction only where SHAP shows a material pair (about 5% or more), and keep it only if it is significant in the GLM.
5. Small factors with a sound rationale but under about 2% permutation importance are left out for parsimony. They are candidates for a later iteration.

**Buildings**

| GLM factor | Treatment | Evidence (SHAP relativity) |
| --- | --- | --- |
| Flood zone | Levels 1-2 (combined), 3, 4, 5, missing | Zones 1 and 2 are both about 0.9. Zone 4 about 1.8, zone 5 about 2.7. |
| Elevation | ≤ 1.8 m, 1.8-3.1 m, 3.1-4.6 m, > 4.6 m | 1.43 → 1.11 → 0.94, then flat at about 0.88 |
| Distance to river | ≤ 0.45 km, 0.45-0.85 km, 0.85-1.3 km, > 1.3 km, missing | 1.13 → 1.04 → 0.96, then flat at about 0.91 |
| Sum insured | log(sum insured), clipped at the 1st/99th percentile | Smooth and monotone, 0.63 → 1.28 across deciles |
| Property age | < 60, 60-85, 85+ years | Flat at about 0.93 up to 58 years, then a step to about 1.10 |
| Dry summer index | ≤ 0.65, 0.65-0.80, > 0.80 | Flat, then +3-4% (subsidence) |
| Prior claims (5y) | 0, 1, 2+ | 0.96, 1.08, 1.24 |
| Region, soil type, energy rating, construction type, credit band, voluntary excess, channel | Categorical (missing as its own level) | Each 1.5-4% permutation importance, with an underwriting rationale |
| **Interaction**: flood zone 4-5 × elevation ≤ 3.1 m | 0/1 indicator | Largest SHAP interaction (5.0%) |

**Contents**

| GLM factor | Treatment | Evidence |
| --- | --- | --- |
| Sum insured | log(sum insured), clipped at the 1st/99th percentile | Smooth, 0.60 → 1.45 |
| Flood zone | 1-2, 3, 4, 5, missing | 0.90, 1.07, 1.62, 3.05 |
| Elevation | ≤ 1.8 m, 1.8-3.1 m, > 3.1 m | 1.32 → 1.01, then flat at about 0.91 |
| Distance to river | ≤ 0.8 km, > 0.8 km, missing | 1.02-1.04, then flat at 0.98 |
| Voluntary excess | 0, 250, 500, 1,000 | 1.04, 1.01, 0.87, 0.88 |
| Prior claims (5y) | 0, 1, 2+ | 0.96, 1.10, 1.13 |
| Accidental damage cover, high-value items, tenure, payment method | Categorical | Cover-driven exposure to risk |
| Region, urban density, property type, occupation | Categorical | Location and household |
| **Interaction**: flood zone 4-5 × elevation ≤ 3.1 m | 0/1 indicator | Largest SHAP interaction (5.6%) |

**Motor**

| GLM factor | Treatment | Evidence |
| --- | --- | --- |
| NCB years | 0-1, 2-3, 4-5, 6-7, 8+ | 1.51 → 1.26 → 0.98 → 0.82, then flat at about 0.77 |
| Cover type | Comprehensive / TPFT / TPO | 26% permutation importance |
| Annual mileage | ≤ 5k, 5-10k, 10-15k, 15-20k, > 20k, missing | 0.82 → 0.95 → 1.14 → 1.21 → 1.27 |
| Driver age | 17-22, 23-28, 29-35, 36-72, 73-80, 81+ | 1.31, 1.11, 0.97, flat about 0.91, 0.93, 0.96 (U-shape) |
| Vehicle group (from make/model) | 5 groups of equal exposure, ranked by a **credibility-weighted** make/model relativity (Z = exposure / (exposure + 1,000)) on 2021-2023 | 300 raw levels are not credible individually. This is the standard actuarial vehicle-grouping approach. |
| Years licensed | 0-2, 3+ | 1.15, then about 0.95 |
| Vehicle age | ≤ 7, 8+ years | 1.03, then 0.94 |
| Telematics | Not opted in, score < 50, 50-65, 65+ | 1.0 → 0.79 as the score improves |
| Prior claims (5y) | 0, 1, 2+ | 0.96, 1.11, 1.22 |
| Voluntary excess, region, channel, payment method, urban density, vehicle use, missed payments (0/1/2+), licence points (0/3/6+) | Categorical | Each 1-7% permutation importance |

No motor interaction is added: the largest pair (NCB × make/model, 4.4%) is below the threshold and would involve the grouped vehicle factor.

**Significance testing.** For each factor, a joint **Wald χ² test** that all of its level coefficients are zero (equivalent to a Type III test), plus coefficient-level 95% confidence intervals. The dispersion parameter is estimated with Pearson χ², as is standard for Tweedie GLMs.

In [0]:
import time
import statsmodels.api as sm
from scipy import stats
from sklearn.metrics import mean_tweedie_deviance

inf = np.inf

def band(x, edges, labels):
    return pd.cut(x.astype(float), edges, labels=labels, include_lowest=True).astype(str).replace('nan', 'missing')

def cat(x):
    return x.astype(str).replace({'nan': 'missing', '<NA>': 'missing', 'None': 'missing'})

def flood(d):
    return d['flood_zone'].map({1: '1-2', 2: '1-2', 3: '3', 4: '4', 5: '5'}).fillna('missing')

# Training-only quantities (2021-2023), so the holdout stays untouched
SI_CLIP = {l: tuple(split(l)[0]['sum_insured'].quantile([0.01, 0.99])) for l in ['buildings', 'contents']}
SI_MED = {l: split(l)[0]['sum_insured'].median() for l in ['buildings', 'contents']}

def vehicle_groups(tr, K=1000):
    g = tr.groupby('make_model', observed=True).agg(exp=('exposure_standardised', 'sum'), cost=(TARGET, 'sum'))
    z = g['exp'] / (g['exp'] + K)
    cred = z * (g['cost'] / g['exp']) / (tr[TARGET].sum() / tr['exposure_standardised'].sum()) + (1 - z)
    order = cred.sort_values().index
    cum = g['exp'].reindex(order).cumsum() / g['exp'].sum()
    return pd.cut(cum, [0, .2, .4, .6, .8, 1.0], labels=['VG1', 'VG2', 'VG3', 'VG4', 'VG5'], include_lowest=True).astype(str), cred
VG_MAP, VG_CRED = vehicle_groups(split('motor')[0])

def log_si(d, line):
    lo, hi = SI_CLIP[line]
    return np.log(d['sum_insured'].clip(lo, hi) / SI_MED[line])

GLM_SPEC = {
    'buildings': lambda d: pd.DataFrame({
        'flood_zone': flood(d),
        'elevation': band(d['elevation_m'], [-inf, 1.8, 3.1, 4.6, inf], ['<=1.8m', '1.8-3.1m', '3.1-4.6m', '>4.6m']),
        'river_distance': band(d['distance_to_river_km'], [-inf, 0.45, 0.85, 1.3, inf], ['<=0.45km', '0.45-0.85km', '0.85-1.3km', '>1.3km']),
        'log_sum_insured': log_si(d, 'buildings'),
        'property_age': band(d['property_age'], [-inf, 60, 85, inf], ['<60', '60-85', '85+']),
        'dry_summer': band(d['dry_summer_index'], [-inf, 0.65, 0.8, inf], ['<=0.65', '0.65-0.80', '>0.80']),
        'prior_claims': cat(d['prior_claims_5y'].clip(upper=2).map({0: '0', 1: '1', 2: '2+'})),
        'region': cat(d['region']), 'soil_type': cat(d['soil_type']), 'energy_rating': cat(d['energy_rating']),
        'construction_type': cat(d['construction_type']), 'credit_band': cat(d['credit_band']),
        'voluntary_excess': cat(d['voluntary_excess']), 'channel': cat(d['channel']),
        'flood45_x_low_elev': ((d['flood_zone'] >= 4) & (d['elevation_m'] <= 3.1)).astype(float)}),
    'contents': lambda d: pd.DataFrame({
        'log_sum_insured': log_si(d, 'contents'),
        'flood_zone': flood(d),
        'elevation': band(d['elevation_m'], [-inf, 1.8, 3.1, inf], ['<=1.8m', '1.8-3.1m', '>3.1m']),
        'river_distance': band(d['distance_to_river_km'], [-inf, 0.8, inf], ['<=0.8km', '>0.8km']),
        'voluntary_excess': cat(d['voluntary_excess']),
        'prior_claims': cat(d['prior_claims_5y'].clip(upper=2).map({0: '0', 1: '1', 2: '2+'})),
        'accidental_damage': cat(d['accidental_damage_cover']),
        'high_value_items': cat(d['high_value_items'].clip(upper=3).map({0: '0', 1: '1', 2: '2', 3: '3+'})),
        'tenure': cat(d['tenure']), 'payment_method': cat(d['payment_method']),
        'region': cat(d['region']), 'urban_density': cat(d['urban_density']),
        'property_type': cat(d['property_type']), 'occupation': cat(d['occupation']),
        'flood45_x_low_elev': ((d['flood_zone'] >= 4) & (d['elevation_m'] <= 3.1)).astype(float)}),
    'motor': lambda d: pd.DataFrame({
        'ncb_years': band(d['ncb_years'], [-inf, 1, 3, 5, 7, inf], ['0-1', '2-3', '4-5', '6-7', '8+']),
        'cover_type': cat(d['cover_type']),
        'annual_mileage': band(d['annual_mileage'], [-inf, 5000, 10000, 15000, 20000, inf], ['<=5k', '5-10k', '10-15k', '15-20k', '>20k']),
        'driver_age': band(d['driver_age'], [-inf, 22, 28, 35, 72, 80, inf], ['17-22', '23-28', '29-35', '36-72', '73-80', '81+']),
        'vehicle_group': d['make_model'].astype(str).map(VG_MAP).fillna('VG3'),
        'years_licensed': band(d['years_licensed'], [-inf, 2, inf], ['0-2', '3+']),
        'vehicle_age': band(d['vehicle_age'], [-inf, 7, inf], ['<=7', '8+']),
        'telematics': np.where(d['telematics_score'].isna(), 'not_opted_in',
                               band(d['telematics_score'], [-inf, 50, 65, inf], ['<50', '50-65', '65+'])),
        'prior_claims': cat(d['prior_claims_5y'].clip(upper=2).map({0: '0', 1: '1', 2: '2+'})),
        'voluntary_excess': cat(d['voluntary_excess']), 'region': cat(d['region']), 'channel': cat(d['channel']),
        'payment_method': cat(d['payment_method']), 'urban_density': cat(d['urban_density']),
        'vehicle_use': cat(d['vehicle_use']),
        'missed_payments': cat(d['missed_payments_12m'].clip(upper=2).map({0: '0', 1: '1', 2: '2+'})),
        # Rows with no customer record are already captured by prior_claims = 'missing'; a second 'missing'
        # level here would be perfectly collinear with it, so they are put in the base level (0 points)
        'licence_points': d['licence_points'].fillna(0).clip(upper=6).map({0: '0', 3: '3', 6: '6+'})}),
}

def design(F, exposure, levels=None):
    """One-hot encode with the largest-exposure level as base; reuse training levels for other data."""
    if levels is None:
        levels = {c: (F.groupby(c)[c].size().index.tolist(),
                      pd.Series(exposure.values, index=F.index).groupby(F[c]).sum().idxmax())
                  for c in F if F[c].dtype == object}
    cols = {'const': np.ones(len(F), dtype=np.float32)}
    for c in F:
        if c in levels:
            lv, base = levels[c]
            for l in lv:
                if l != base:
                    cols[f'{c}[{l}]'] = (F[c] == l).values.astype(np.float32)
        else:
            cols[c] = F[c].values.astype(np.float32)
    return pd.DataFrame(cols, index=F.index), levels

class TweedieGLM:
    """Memory-light IRLS for a Tweedie GLM with log link and offset (statsmodels needs several n x p copies,
    which exceeds memory at ~700k rows). Fisher scoring: weights mu^(2-p), working response eta - offset + (y - mu)/mu.
    Dispersion = Pearson chi2 / (n - k); covariance = dispersion x (X'WX)^-1. Checked against statsmodels below."""
    def __init__(self, p=1.5, tol=1e-9, max_iter=50, chunk=100_000):
        self.p, self.tol, self.max_iter, self.chunk = p, tol, max_iter, chunk

    def _xtwx(self, X, w, z=None):
        k = X.shape[1]; A = np.zeros((k, k)); b = np.zeros(k)
        for i in range(0, len(X), self.chunk):
            Xi = X[i:i + self.chunk].astype(np.float64); wi = w[i:i + self.chunk]
            A += (Xi * wi[:, None]).T @ Xi
            if z is not None:
                b += Xi.T @ (wi * z[i:i + self.chunk])
        return A, b

    def fit(self, X, y, offset):
        Xv, y, off, p = X.to_numpy(np.float32), np.asarray(y, float), np.asarray(offset, float), self.p
        beta = np.zeros(Xv.shape[1]); beta[0] = np.log(y.sum() / np.exp(off).sum())
        dev_old = np.inf
        for it in range(self.max_iter):
            eta = Xv @ beta.astype(np.float32) + off
            mu = np.exp(eta)
            w = mu ** (2 - p)
            A, b = self._xtwx(Xv, w, eta - off + (y - mu) / mu)
            beta = np.linalg.solve(A, b)
            mu = np.exp(Xv @ beta.astype(np.float32) + off)
            dev = mean_tweedie_deviance(y, mu, power=p) * len(y)
            if abs(dev_old - dev) / dev < self.tol:
                break
            dev_old = dev
        self.n_iter, self.deviance, self.nobs = it + 1, dev, len(y)
        self.scale = np.sum((y - mu) ** 2 / mu ** p) / (len(y) - len(beta))
        A, _ = self._xtwx(Xv, mu ** (2 - p))
        cov = self.scale * np.linalg.inv(A)
        names = X.columns
        self.params = pd.Series(beta, index=names)
        self.bse = pd.Series(np.sqrt(np.diag(cov)), index=names)
        self.cov = pd.DataFrame(cov, index=names, columns=names)
        self.pvalues = pd.Series(2 * stats.norm.sf(np.abs(beta / self.bse.values)), index=names)
        return self

    def conf_int(self, z=1.96):
        return pd.DataFrame({0: self.params - z * self.bse, 1: self.params + z * self.bse})

    def predict(self, X, offset):
        return np.exp(X.to_numpy(np.float32) @ self.params.values.astype(np.float32) + np.asarray(offset, float))

# Verification on a 50k-row buildings sample: our IRLS must reproduce statsmodels' coefficients, SEs and dispersion
_tr = split('buildings')[0].sample(50_000, random_state=1)
_X, _ = design(GLM_SPEC['buildings'](_tr), _tr['exposure_standardised'])
_X = _X.loc[:, _X.sum() > 0]
_sm = sm.GLM(_tr[TARGET].values, _X.astype(float), offset=np.log(_tr['exposure_standardised'].values),
             family=sm.families.Tweedie(var_power=1.5, link=sm.families.links.Log())).fit(scale='X2')
_own = TweedieGLM().fit(_X, _tr[TARGET], np.log(_tr['exposure_standardised']))
print('Check vs statsmodels (50k sample): max |coef diff| = %.2e, max |SE diff| / SE = %.2e, dispersion %.2f vs %.2f'
      % (np.abs(_own.params.values - _sm.params.values).max(), (np.abs(_own.bse.values - _sm.bse.values) / _sm.bse.values).max(),
         _own.scale, _sm.scale))
assert np.abs(_own.params.values - _sm.params.values).max() < 1e-4
del _tr, _X, _sm, _own

glms, glm_tables, glm_tests = {}, [], []
for line in LINES:
    tr, ho = split(line)
    Ftr = GLM_SPEC[line](tr)
    Xtr, levels = design(Ftr, tr['exposure_standardised'])
    t0 = time.time()
    res = TweedieGLM().fit(Xtr, tr[TARGET], np.log(tr['exposure_standardised']))
    glms[line] = {'res': res, 'levels': levels, 'columns': Xtr.columns}
    # Coefficient / relativity table
    ci = res.conf_int()
    exp_share = {}
    for c in Ftr:
        if c in levels:
            s = tr['exposure_standardised'].groupby(Ftr[c]).sum(); exp_share.update({f'{c}[{k}]': v / s.sum() for k, v in s.items()})
    t = pd.DataFrame({'line': line, 'term': res.params.index, 'coef': res.params.values, 'se': res.bse.values,
                      'relativity': np.exp(res.params.values), 'rel_ci_low': np.exp(ci[0].values),
                      'rel_ci_high': np.exp(ci[1].values), 'p_value': res.pvalues.values})
    t['factor'] = t['term'].str.split('[', regex=False).str[0]
    t['level'] = t['term'].str.extract(r'\[(.*)\]')[0]
    t['exposure_share'] = t['term'].map(exp_share)
    glm_tables.append(t)
    # Joint Wald chi-square test per factor (Type III equivalent)
    for f in Ftr:
        idx = [c for c in Xtr.columns if c == f or c.startswith(f + '[')]
        b = res.params[idx].values; V = res.cov.loc[idx, idx].values
        chi2 = float(b @ np.linalg.solve(V, b))
        glm_tests.append({'line': line, 'factor': f, 'df': len(idx), 'base_level': levels[f][1] if f in levels else '(numeric)',
                          'wald_chi2': chi2, 'p_value': stats.chi2.sf(chi2, len(idx))})
    print(f'{line}: {Xtr.shape[1]} parameters, {res.nobs:,.0f} rows, {res.n_iter} IRLS iterations, dispersion {res.scale:.1f}, '
          f'base rate {np.exp(res.params["const"]):,.1f} CU per year, {time.time() - t0:.0f}s')
    del Xtr, Ftr

glm_tables = pd.concat(glm_tables, ignore_index=True)
glm_tests = pd.DataFrame(glm_tests)
glm_tests['significant_1%'] = glm_tests['p_value'] < 0.01
display(glm_tests.round({'wald_chi2': 1}).assign(p_value=glm_tests['p_value'].map(lambda p: f'{p:.2e}')))

### GLM significance results, and refinement

The custom IRLS fitter reproduces statsmodels on a 50,000-row check sample (maximum coefficient difference 2.7e-5, standard errors within 5e-6 relative, identical dispersion). It is used for the full fits because statsmodels runs out of memory at about 700,000 rows.

The joint Wald tests (table above) confirm most of the XGBoost-selected factors at the 1% level, with very large χ² for the main drivers (e.g. motor NCB χ² = 682 on 4 df, buildings sum insured 317 on 1 df, buildings flood zone 117 on 4 df). Some candidates are **not** significant once the others are in the model. This is where the GLM adds discipline to what XGBoost found:

| Line | Not significant at 1% | Interpretation |
| --- | --- | --- |
| Buildings | river distance (p = 0.13), dry summer (p = 0.04), flood zone 4-5 × low elevation interaction (p = 0.90) | River distance carries no information beyond flood zone and elevation: they measure the same flood exposure. The SHAP "interaction" is already captured by the non-linear bands for flood zone and elevation. |
| Contents | river distance (p = 0.30), tenure (p = 0.06), property type (p = 0.07), occupation (p = 0.57, 20 df), interaction (p = 0.13) | As for buildings. Occupation's small XGBoost signal does not survive a formal test with 20 parameters. |
| Motor | none | All 17 factors are significant at 1% |

**Refinement rule:** remove factors with p ≥ 1% and refit (one pass of backward elimination). The result is a **final GLM**, in which every factor is significant and justifiable.

In [0]:
def fit_line(line, drop=()):
    tr, _ = split(line)
    F = GLM_SPEC[line](tr).drop(columns=list(drop))
    X, levels = design(F, tr['exposure_standardised'])
    res = TweedieGLM().fit(X, tr[TARGET], np.log(tr['exposure_standardised']))
    ci = res.conf_int()
    exp_share = {}
    for c in F:
        if c in levels:
            s = tr['exposure_standardised'].groupby(F[c]).sum()
            exp_share.update({f'{c}[{k}]': v / s.sum() for k, v in s.items()})
    table = pd.DataFrame({'line': line, 'term': res.params.index, 'coef': res.params.values, 'se': res.bse.values,
                          'relativity': np.exp(res.params.values), 'rel_ci_low': np.exp(ci[0].values),
                          'rel_ci_high': np.exp(ci[1].values), 'p_value': res.pvalues.values})
    table['factor'] = table['term'].str.split('[', regex=False).str[0]
    table['level'] = table['term'].str.extract(r'\[(.*)\]')[0]
    table['exposure_share'] = table['term'].map(exp_share)
    tests = []
    for f in F:
        idx = [c for c in X.columns if c == f or c.startswith(f + '[')]
        b = res.params[idx].values; V = res.cov.loc[idx, idx].values
        chi2 = float(b @ np.linalg.solve(V, b))
        tests.append({'line': line, 'factor': f, 'df': len(idx), 'base_level': levels[f][1] if f in levels else '(numeric)',
                      'base_exposure_share': exp_share.get(f'{f}[{levels[f][1]}]') if f in levels else None,
                      'wald_chi2': chi2, 'p_value': stats.chi2.sf(chi2, len(idx))})
    return {'res': res, 'levels': levels, 'drop': list(drop), 'table': table, 'tests': pd.DataFrame(tests)}

GLM_DROP = {l: glm_tests.query('line == @l and p_value >= 0.01')['factor'].tolist() for l in LINES}
print('Dropped (p >= 1% in the candidate model):', GLM_DROP)
glm_final = {l: fit_line(l, GLM_DROP[l]) for l in LINES}

final_tests = pd.concat([glm_final[l]['tests'] for l in LINES], ignore_index=True)
final_tests['significant_1%'] = final_tests['p_value'] < 0.01
display(final_tests.round({'wald_chi2': 1, 'base_exposure_share': 3}).assign(p_value=final_tests['p_value'].map(lambda p: f'{p:.1e}')))
for l in LINES:
    r = glm_final[l]['res']
    print(f"{l}: {len(r.params)} parameters, base rate (all factors at base level) {np.exp(r.params['const']):,.1f} CU per policy year")

### Final GLM relativity tables

The charts below are the board-facing output: one panel per rating factor, showing the fitted **relativity for each level** (bars) with its **95% confidence interval** (black whiskers), against the **share of exposure** in each level (grey line, right axis). The base level is 1.00 by construction.

* A whisker that crosses 1.00 means the level is not statistically different from the base level. It is a candidate to merge with the base in a later simplification.
* Wide whiskers over a small exposure share show where the data is thin. In practice these levels are smoothed or capped using underwriting judgement before implementation.
* `log_sum_insured` is a continuous term, shown as a curve: relativity = (sum insured / median)^β.

The full coefficient table (coefficient, standard error, relativity, CI, p-value and exposure share per level) is displayed under the charts, for audit.

In [0]:
for line in LINES:
    g = glm_final[line]; t = g['table']; levels = g['levels']
    factors = [f for f in g['tests']['factor']]
    ncol = 4; nrow = int(np.ceil(len(factors) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(20, 4.2 * nrow)); axes = axes.ravel()
    for ax, f in zip(axes, factors):
        if f in levels:
            lv, base = levels[f]
            rows = [{'level': l, 'rel': 1.0, 'lo': 1.0, 'hi': 1.0, 'exp': t.loc[t['term'] == f'{f}[{l}]', 'exposure_share'].sum()}
                    if l == base else
                    {'level': l, **t.loc[t['term'] == f'{f}[{l}]', ['relativity', 'rel_ci_low', 'rel_ci_high', 'exposure_share']]
                        .set_axis(['rel', 'lo', 'hi', 'exp'], axis=1).iloc[0].to_dict()} for l in lv]
            d = pd.DataFrame(rows)
            tr, _ = split(line)
            s = tr['exposure_standardised'].groupby(GLM_SPEC[line](tr)[f]).sum(); d['exp'] = d['level'].map(s / s.sum())
            x = np.arange(len(d))
            ax.bar(x, d['rel'], color=np.where(d['level'] == base, 'tab:green', 'tab:blue'), alpha=0.85)
            ax.errorbar(x, d['rel'], yerr=[d['rel'] - d['lo'], d['hi'] - d['rel']], fmt='none', ecolor='black', capsize=3)
            ax.set_xticks(x); ax.set_xticklabels(d['level'], rotation=45, ha='right', fontsize=8)
            ax2 = ax.twinx(); ax2.plot(x, d['exp'], color='grey', marker='o', ms=3); ax2.set_ylim(0, max(1, d['exp'].max() * 1.1))
            ax2.set_ylabel('exposure share', fontsize=8, color='grey')
        else:
            b = g['res'].params[f]; se = g['res'].bse[f]
            lo, hi = SI_CLIP[line]; si = np.linspace(lo, hi, 100); z = np.log(si / SI_MED[line])
            ax.plot(si, np.exp(b * z), color='tab:blue', lw=2)
            ax.fill_between(si, np.exp((b - 1.96 * se) * z), np.exp((b + 1.96 * se) * z), color='tab:blue', alpha=0.2)
            ax.set_xlabel('sum insured (CU)'); f = f'{f}: β = {b:.3f} (SE {se:.3f})'
        ax.axhline(1, color='black', lw=0.7); ax.set_title(f, fontsize=11); ax.set_ylabel('relativity')
    for ax in axes[len(factors):]: ax.axis('off')
    fig.suptitle(f'{line.title()}: final GLM relativities with 95% confidence intervals (green = base level)', fontsize=15)
    plt.tight_layout(); plt.show()

glm_coef_table = pd.concat([glm_final[l]['table'] for l in LINES], ignore_index=True)
display(glm_coef_table[['line', 'factor', 'level', 'coef', 'se', 'relativity', 'rel_ci_low', 'rel_ci_high', 'p_value', 'exposure_share']].round(4))

## Step 7: How much accuracy does explainability cost? GLM vs XGBoost on the 2024 holdout

All models were fitted on 2021-2023 only, so 2024 is a genuine out-of-time test. Three measures are used:

* **Tweedie D²** (Kaggle metric), raw and level-adjusted. Level-adjusted rescales each model's 2024 predictions to the 2024 actual total, so the comparison is about **risk ranking** rather than the overall level (which depends on trend and IBNR, #16).
* **Gini coefficient**: policies are sorted from lowest to highest predicted cost per exposure year, and the Gini is twice the area between the cumulative actual-cost curve and the diagonal. 0 means no ranking power; higher is better. It does not depend on the overall level.
* **Lift chart**: 2024 policies are split into 10 equal-exposure bands by predicted rate. In each band, actual and predicted loss cost are compared, both as a ratio to the overall average. A good model shows a steep, well-calibrated staircase.
* **Double-lift chart**: policies are sorted by the ratio of the GLM to the XGBoost prediction. Where the two models disagree, the chart shows which one the actual experience supports. If the actual line follows XGBoost, the GLM is missing a signal there.

In [0]:
import matplotlib.pyplot as plt

def gini(actual, pred, exposure):
    o = np.argsort(pred / exposure)
    cum_e = np.concatenate([[0], np.cumsum(exposure[o]) / exposure.sum()])
    cum_a = np.concatenate([[0], np.cumsum(actual[o]) / actual.sum()])
    return 1 - 2 * np.trapezoid(cum_a, cum_e)

def glm_predict(g, line, df):
    X, _ = design(GLM_SPEC[line](df).drop(columns=g['drop']), df['exposure_standardised'], levels=g['levels'])
    return g['res'].predict(X, np.log(df['exposure_standardised']))

glm_candidate = {l: {'res': glms[l]['res'], 'levels': glms[l]['levels'], 'drop': []} for l in LINES}
comparison, lift_data = [], {}
for line in LINES:
    _, ho = split(line)
    h = holdout[line].copy()
    assert (h['row_id'].values == ho['row_id'].values).all()
    h['pred_glm_full'] = glm_predict(glm_candidate[line], line, ho)
    h['pred_glm'] = glm_predict(glm_final[line], line, ho)
    y, e = h[TARGET].to_numpy(), h['exposure_standardised'].to_numpy()
    for name, col in [('XGBoost (benchmark)', 'pred_xgb'), ('GLM candidate (all factors)', 'pred_glm_full'), ('GLM final (significant factors)', 'pred_glm')]:
        p = h[col].to_numpy(); d2, d2a = d2_scores(h[TARGET], p)
        comparison.append({'line': line, 'model': name, 'parameters': {'pred_xgb': None, 'pred_glm_full': len(glm_candidate[line]['res'].params),
                                                                       'pred_glm': len(glm_final[line]['res'].params)}[col],
                           'D2': d2, 'D2_level_adj': d2a, 'gini': gini(y, p, e), 'pred/actual 2024': p.sum() / y.sum()})
    lift_data[line] = h
comparison = pd.DataFrame(comparison)
xgb_ref = comparison[comparison['model'].str.startswith('XGB')].set_index('line')['D2_level_adj']
comparison['% of XGBoost D2 (level-adj)'] = comparison['D2_level_adj'] / comparison['line'].map(xgb_ref) * 100
display(comparison.round(4))

# Lift charts (equal-exposure deciles of predicted rate) and double-lift charts
fig, axes = plt.subplots(2, 3, figsize=(19, 10))
for i, line in enumerate(LINES):
    h = lift_data[line]; e = h['exposure_standardised']
    for col in ['pred_xgb', 'pred_glm']:
        h[col + '_adj'] = h[col] * h[TARGET].sum() / h[col].sum()
    avg = h[TARGET].sum() / e.sum()
    # Lift: deciles of GLM predicted rate
    o = h.assign(rate=h['pred_glm'] / e).sort_values('rate')
    o['band'] = np.minimum((o['exposure_standardised'].cumsum() / e.sum() * 10).astype(int), 9) + 1
    g = o.groupby('band').agg(exp=('exposure_standardised', 'sum'), act=(TARGET, 'sum'), glm=('pred_glm_adj', 'sum'), xgb=('pred_xgb_adj', 'sum'))
    ax = axes[0, i]
    ax.bar(g.index, g['act'] / g['exp'] / avg, color='lightgrey', label='Actual 2024')
    ax.plot(g.index, g['glm'] / g['exp'] / avg, 'o-', color='tab:blue', label='GLM final')
    ax.plot(g.index, g['xgb'] / g['exp'] / avg, 's--', color='tab:orange', label='XGBoost')
    ax.set_title(f'{line.title()}: lift chart (deciles of GLM predicted rate)'); ax.set_xlabel('decile'); ax.set_ylabel('loss cost / average'); ax.legend()
    # Double lift: deciles of GLM / XGBoost ratio
    o = h.assign(ratio=h['pred_glm_adj'] / h['pred_xgb_adj']).sort_values('ratio')
    o['band'] = np.minimum((o['exposure_standardised'].cumsum() / e.sum() * 10).astype(int), 9) + 1
    g = o.groupby('band').agg(act=(TARGET, 'sum'), glm=('pred_glm_adj', 'sum'), xgb=('pred_xgb_adj', 'sum'))
    ax = axes[1, i]
    ax.plot(g.index, g['act'] / g['xgb'], 'o-', color='black', label='Actual / XGBoost')
    ax.plot(g.index, g['glm'] / g['xgb'], 's--', color='tab:blue', label='GLM / XGBoost')
    ax.axhline(1, color='tab:orange', lw=1, label='XGBoost (=1)')
    ax.set_title(f'{line.title()}: double lift (deciles of GLM / XGBoost)'); ax.set_xlabel('decile'); ax.set_ylabel('ratio to XGBoost'); ax.legend()
plt.tight_layout(); plt.show()

## Conclusions

### Results on the 2024 holdout (capped cost, all models fitted on 2021-2023)

| Line | XGBoost D² | Final GLM D² | GLM as % of XGBoost | XGBoost Gini | GLM Gini | GLM parameters |
| --- | --- | --- | --- | --- | --- | --- |
| Buildings | 0.101 | 0.107 | 106% | 0.339 | 0.369 | 49 |
| Contents | 0.077 | 0.082 | 107% | 0.321 | 0.330 | 33 |
| Motor | 0.082 (level-adj) | 0.090 (level-adj) | 110% | 0.272 | 0.299 | 56 |

**On this data, explainability costs nothing.** The final GLMs match or slightly beat the XGBoost benchmark on every line, on both D² and Gini. This is the result to take to the board, with the following caveats:

* The XGBoost models use fixed, untuned hyperparameters, and early stopping chose only 67-134 trees. A tuned XGBoost would probably close some of the gap. The conclusion is therefore "a well-specified GLM is **at least as good** as an off-the-shelf GBM here", not that GLMs are better in general.
* The GLM benefited from XGBoost. Its factors, bands and the decision to drop interactions all came from the SHAP and permutation analysis. That is the intended workflow: **machine learning for discovery, GLM for production**.
* Early stopping for XGBoost used the 2024 holdout, which, if anything, flatters XGBoost.

### What a board member needs to know

1. **Price is driven by a small set of intuitive factors.** For home, these are flood exposure (flood zone and elevation) and the amount insured. For motor, they are no-claims bonus, cover type, mileage and driver age. Each is shown as a relativity table with confidence intervals (Step 6 charts).
2. **Every factor in the final GLMs is statistically significant at the 1% level** (joint Wald tests) and has an underwriting rationale. Eight non-significant candidates were removed, including river distance, which adds nothing beyond flood zone and elevation.
3. **Factors were excluded deliberately, not by accident:**
   * `customer_service_contacts` is excluded as target leakage: it records claims as they happen, and the 2025 data confirms this.
   * `written_premium` is excluded because it is the old price, not a measure of risk.
   * Marketing and operational fields (`previous_insurer`, `branch_id`, `campaign_code`, `marketing_segment` etc.) are excluded on governance grounds. They are predictive but hard to defend as rating factors. **The pricing committee should confirm this choice.**
4. **Missing data carries risk.** Buildings rows with a missing flood zone have a relativity of 2.03 (95% CI 1.71-2.40), worse than zone 4. That points to a data-capture issue worth raising with underwriting. Until it is fixed, a missing flood zone is priced as high risk, not average.

### Limitations and next steps

* **Base level, trend and development.** Relativities are fitted on undeveloped claims. The 2025 base rate still needs chain-ladder IBNR/IBNER development (#16) and a trend allowance (buildings loss cost is rising about 5% a year), and the large-loss loading must be applied (1.044 buildings, 1.022 contents, 1.097 motor).
* **Open questions for data owners:** what the construction codes `T` and `W` mean, why flood zone is missing for 2.9% of addresses, and why about 10% of motor rows have no customer record (priced through the `prior_claims = missing` level, relativity 1.15).
* **Possible refinements:** peril-level models (e.g. separate flood and escape-of-water models for buildings), merging levels whose CIs overlap 1.00 (several regions), smoothing thin levels, and a tuned XGBoost as a stronger benchmark.
* **Submission v003:** refit the final GLMs on 2021-2024, apply development, trend and large-loss loadings, and score the 2025 test files.

## Submissions v003 (final GLM) and v004 (XGBoost): leaderboard test of risk differentiation

These two files test on the Kaggle leaderboard whether the models' **ranking of risks** beats the constant rate in v002.

* **v003**: the final GLMs (Step 6, significant factors only) applied to the 2025 test rows.
* **v004**: the XGBoost benchmarks (Step 3) applied to the same rows.

**Same overall level as v002, on purpose.** Each line's predictions are rescaled so that the line's total equals v002's total: the v002 rate (buildings 420.28, contents 111.92, motor 548.46 CU per year) × 2025 test exposure. The **only** difference from v002 is how cost is shared between policies. Any change in score is therefore caused by the risk relativities, not by a change in level.

This is needed because the models were fitted on **capped** cost for 2021-2023. On its own, that level understates the target, which is uncapped, ultimate and 2025. Proper level work (large-loss loading, IBNR development, trend; #16) is the next step. It will be tested as a separate submission so its effect can be seen on its own.

The file checks are the same as v002: 626,639 rows in `sample_submission` order, no nulls, and every prediction > 0 (floored at 0.01). Only the CSV is saved.

In [0]:
import os

V002_RATE = {'buildings': 420.28, 'contents': 111.92, 'motor': 548.46}  # CU per policy year (v002)
sub_template = kw_data_typed['sample_submission'][['row_id']]
out_dir = '../submissions'

parts = {'v003': [], 'v004': []}
level_check = []
for line in LINES:
    te = frames[line]['test']
    e = te['exposure_standardised'].to_numpy()
    raw = {
        'v003': np.asarray(glm_predict(glm_final[line], line, te), dtype=float),
        'v004': xgb_models[line]['booster'].predict(dmatrix(te, line, with_label=False),
                                                   iteration_range=xgb_models[line]['iteration_range']),
    }
    target_total = V002_RATE[line] * e.sum()
    for v, p in raw.items():
        assert np.isfinite(p).all() and (p > 0).all(), f'{v} {line}: non-finite or non-positive raw prediction'
        scale = target_total / p.sum()
        parts[v].append(pd.DataFrame({'row_id': te['row_id'].values, 'prediction': p * scale}))
        rate = p * scale / e
        level_check.append({'version': v, 'line': line, 'rows': len(te), 'scale_to_v002_level': scale,
                            'total_CU': p.sum() * scale, 'v002_total_CU': target_total,
                            'rate_p5': np.percentile(rate, 5), 'rate_median': np.median(rate), 'rate_p95': np.percentile(rate, 95)})

subs = {}
for v, lst in parts.items():
    sub = sub_template.merge(pd.concat(lst), on='row_id', how='left', validate='one_to_one')
    sub['prediction'] = sub['prediction'].clip(lower=0.01).astype('float64').round(2)
    assert len(sub) == 626_639 and sub['row_id'].is_unique
    assert (sub['row_id'].values == sub_template['row_id'].values).all()
    assert sub['prediction'].notna().all() and (sub['prediction'] > 0).all()
    path = f'{out_dir}/home_insurance_submission_{v}.csv'
    sub.to_csv(path, index=False, float_format='%.2f')
    # Re-read from disk to confirm what Kaggle will receive
    chk = pd.read_csv(path, dtype={'row_id': str})
    assert chk['row_id'].equals(sub_template['row_id']) and (chk['prediction'] > 0).all()
    subs[v] = sub
    print(f'Saved {len(chk):,} rows to {os.path.abspath(path)} | min {chk.prediction.min():.2f}, mean {chk.prediction.mean():.2f}')

display(pd.DataFrame(level_check).round(3))

# How different are the models from each other and from v002? (correlation of predicted rates)
v002 = pd.read_csv(f'{out_dir}/home_insurance_submission_v002.csv', dtype={'row_id': str})
assert v002['row_id'].equals(sub_template['row_id'])
cmp = pd.DataFrame({'line': sub_template['row_id'].str[0].map(policy_type_map), 'v002': v002['prediction'],
                    'v003': subs['v003']['prediction'], 'v004': subs['v004']['prediction']})
display(cmp.groupby('line')[['v002', 'v003', 'v004']].sum().round(0).reset_index())
display(cmp.groupby('line').apply(lambda d: pd.Series({'corr v003 vs v004': d['v003'].corr(d['v004'])})).round(3).reset_index())

## Submission v005: GLM/XGBoost blend, with a 2025 cost level

v005 improves on v003/v004 in two ways:

1. **Risk ranking: blend the GLM and XGBoost.** Their predictions correlate at only 0.80-0.88, so they make partly different errors. The blend weight per line is chosen on the 2024 holdout (level-adjusted D², on a grid from 0 to 1). This flatters the blend slightly, because XGBoost early-stopped on the same holdout.
2. **Cost level: move from the 2021-23 capped level to the expected 2025 uncapped level.** Both models predict at the **2021-2023 capped** level. Per line, v005 multiplies by:
   * the **large-loss loading** = uncapped / capped cost for 2021-2023, which restores the losses above the 99.5% cap;
   * a **trend to 2025** = (1 + g)^3, where g is the log-linear annual change in capped loss cost per exposure year over 2021-2023. 2024 is left out because it is still under-reported (IBNR). The factor is 3 years because the training midpoint is 2022.

**Limitations.** There is no chain-ladder development yet (#16), so any IBNR still left in 2021-2023 is not included and v005 probably still understates the level slightly. The trend comes from only three points, so g is limited to 0% to +8% a year. Motor's raw trend is -2.5% a year. That is most likely because 2023 is less developed than 2021, not a real fall in cost, so motor gets no trend.

In [0]:
W_GRID = np.round(np.arange(0, 1.01, 0.1), 1)
v005_parts, v005_info = [], []
for line in LINES:
    # 1. Blend weight on the 2024 holdout (w = GLM weight)
    h = lift_data[line]
    y = h[TARGET]
    scores = {w: d2_scores(y, w * h['pred_glm'] + (1 - w) * h['pred_xgb'])[1] for w in W_GRID}
    w = max(scores, key=scores.get)

    # 2. Level: large-loss loading and trend, from 2021-2023 only
    tr, _ = split(line)
    ll = tr['cost'].sum() / tr[TARGET].sum()
    yr = tr.groupby('policy_year').agg(c=(TARGET, 'sum'), e=('exposure_standardised', 'sum'))
    lc = yr['c'] / yr['e']
    g_raw = np.exp(np.polyfit(lc.index.values, np.log(lc.values), 1)[0]) - 1
    # Floor at 0%: a fall over 2021-23 (motor) is most likely the latest year being less developed, not a real fall in cost
    g = float(np.clip(g_raw, 0.0, 0.08))
    trend = (1 + g) ** (2025 - 2022)

    # 3. Score 2025 test rows (both models at the 2021-23 capped level)
    te = frames[line]['test']
    p_glm = np.asarray(glm_predict(glm_final[line], line, te), dtype=float)
    xm = xgb_models[line]
    p_xgb = xm['booster'].predict(dmatrix(te, line, with_label=False), iteration_range=xm['iteration_range']) * xm['balance_factor']
    p = (w * p_glm + (1 - w) * p_xgb) * ll * trend
    assert np.isfinite(p).all() and (p > 0).all()
    v005_parts.append(pd.DataFrame({'row_id': te['row_id'].values, 'prediction': p}))
    e = te['exposure_standardised'].sum()
    v005_info.append({'line': line, 'GLM_weight': w, 'D2_GLM_2024': scores[1.0], 'D2_XGB_2024': scores[0.0], 'D2_blend_2024': scores[w],
                      'loss_cost_2021': lc.iloc[0], 'loss_cost_2023': lc.iloc[-1], 'trend_g_raw': g_raw, 'trend_g_used': g,
                      'large_loss_loading': ll, 'trend_factor': trend, 'v005_rate_CU': p.sum() / e, 'v002_rate_CU': V002_RATE[line]})

v005_info = pd.DataFrame(v005_info)
display(v005_info.round(4))

sub = sub_template.merge(pd.concat(v005_parts), on='row_id', how='left', validate='one_to_one')
sub['prediction'] = sub['prediction'].clip(lower=0.01).astype('float64').round(2)
assert len(sub) == 626_639 and sub['row_id'].is_unique and (sub['row_id'].values == sub_template['row_id'].values).all()
assert sub['prediction'].notna().all() and (sub['prediction'] > 0).all()
path = f'{out_dir}/home_insurance_submission_v005.csv'
sub.to_csv(path, index=False, float_format='%.2f')
chk = pd.read_csv(path, dtype={'row_id': str})
assert chk['row_id'].equals(sub_template['row_id']) and (chk['prediction'] > 0).all() and chk['prediction'].notna().all()
print(f'Saved {len(chk):,} rows to {os.path.abspath(path)} | min {chk.prediction.min():.2f}, mean {chk.prediction.mean():.2f}')
display(chk.assign(line=chk['row_id'].str[0].map(policy_type_map)).groupby('line')['prediction'].agg(['count', 'sum', 'mean']).round(2).reset_index())